# Make the Loop Tell the Truth — one training step, instrumented end to end

**Assignment 10 — the training loop.** *"Take a small model and a real loop, and make it
tell you the truth about itself. […] Print things and check things. Every serious
training bug is silent, and the loss curve is not going to be the one that tells you."*

Session 9 ended with one scalar — the loss. This notebook is about the step that scalar
triggers, and about the instruments that tell you whether the step is doing what you
think it is. Six required tasks, each measured on a real (tiny) model and a real loop:

| # | task | section |
|---|---|---|
| 1 | print **every tensor shape** in one step; one line per dimension | §3 |
| 2 | verify **one gradient by hand** — nudge, measure, compare with `backward()` | §4 |
| 3 | **break gradient accumulation on purpose** — average of averages, micro-batches of different lengths, both curves plotted | §5 |
| 4 | log the **grad norm at every step**; find one step where it moved **before** the loss | §6 |
| 5 | compute my own **MFU**, honestly, and say what costs the distance to 40% | §7 |
| 6 | **0.1 in fp32, bf16, fp8 E4M3**, bits written by hand; pick a training precision | §8 |

Everything is deterministic (seeded, CPU), self-contained (no downloads), and every
claimed number is written to `submission_artifacts/`, where `audit.py` re-derives it
from disk without executing this notebook. Machine-specific timings are recorded but
never audited verbatim — only their arithmetic is (the Session-9 convention).

## §0 — Setup

One config, one artifact dir, one formatter. `A10_FAST=1` shrinks every budget for a
smoke run; the committed artifacts always come from a full run.

In [1]:
import copy
import gc
import hashlib
import json
import math
import os
import random
import struct
import time
from fractions import Fraction
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

FAST = os.environ.get("A10_FAST", "") == "1"
SEED = 1337
DEVICE = "cpu"          # the point of this notebook is CPU-honest measurement
N_THREADS = min(4, os.cpu_count() or 1)
torch.set_num_threads(N_THREADS)
try:
    torch.set_float32_matmul_precision("highest")   # same math in the loop and the GEMM proxy
except AttributeError:
    pass

CFG = {
    "vocab": 259, "d_model": 128, "n_layer": 2, "n_head": 4, "max_pos": 256,
    "T": 128,                             # context length (sequence positions)
    "B": 8,                               # sequences per (full) batch
    "lr": 3e-3,                           # AdamW, used everywhere except where stated
    "hz_d": 64, "hz_T": 40,               # SS5 hazard-corpus model width / padded length
    "steps_accum": 80 if FAST else 400,   # per gradient-accumulation arm
    "accum_K": 4,                         # micro-batches per global batch in SS5
    "accum_seeds": [1337] if FAST else [1337, 1338, 1339],
    "steps_norm": 100 if FAST else 300,   # per grad-norm arm in SS6
    "inj_step": 50 if FAST else 150,      # the anomalous batch lands here
    "eval_every": 4 if FAST else 10,
    "mfu_warmup": 3 if FAST else 8,
    "mfu_steps": 10 if FAST else 30,      # timed steps for tokens/sec
    "gemm_sizes": [512, 1024] if FAST else [1024, 2048, 4096],
    "widths": [64, 128] if FAST else [64, 128, 256, 512],
}

ART = Path(os.environ.get("A10_ART_DIR", "submission_artifacts"))
(ART / "plots").mkdir(parents=True, exist_ok=True)
RESULTS = {"config": {**CFG, "seed": SEED, "device": DEVICE, "fast": FAST,
                      "torch": torch.__version__, "threads": N_THREADS,
                      "cpu_count": os.cpu_count()}}
HEADLINE = {}   # name -> formatted string; README quotes these verbatim
CURVES = {}     # per-step traces, saved for the independent audit
T0 = time.time()


def fmt(x, nd=4):
    """One formatter for every float that reaches the README (verbatim-audited)."""
    return f"{x:.{nd}f}"


def seed_all(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)


def strict_assert(cond, msg=""):
    """Training-outcome assertions: hard in a full run, a loud warning under the
    reduced A10_FAST budgets (where curves have not had time to separate)."""
    if FAST and not cond:
        print(f"[FAST] skipped assert: {msg}")
        return
    assert cond, msg


def sha(obj):
    """Checksum for 'the arms saw identical data' assertions."""
    return hashlib.sha256(repr(obj).encode()).hexdigest()[:16]


In [2]:
cpu_name = "unknown"
try:
    for line in open("/proc/cpuinfo"):
        if line.startswith("model name"):
            cpu_name = line.split(":", 1)[1].strip(); break
except OSError:
    pass
RESULTS["config"]["cpu_name"] = cpu_name
print(f"device={DEVICE}  torch={torch.__version__}  threads={N_THREADS}  fast={FAST}")
print(f"cpu: {cpu_name}")

device=cpu  torch=2.13.0+cpu  threads=4  fast=False
cpu: Intel(R) Xeon(R) Processor @ 2.80GHz


## §1 — Tokens you can read: a byte-level tokenizer and the corpora

Byte-level, V = 259 (`<pad>/<bos>/<eos>` + all 256 bytes) — every id decodes to a
printable string, and ln(259) = 5.5568 gives the untrained model a known starting loss.

Three data sources, each built for the section that needs it:

- **prose** — ten self-authored documents *about this session* (the main corpus for
  §3/§4/§6/§7);
- **telemetry** — very short log-style lines. §5d mixes them with prose to test whether
  a "different register" makes the accumulation bug visible (spoiler, measured below:
  barely — that negative result is reported, not hidden);
- **the hazard corpus** (§5c) — documents that are `<bos> aa…a <eos>` with length 2 or
  32, 50/50. Every position is deterministic *except one*: after two `a`s the document
  either continues or ends, with true probability ½. That single decision is invisible
  to context (both document types share the prefix), so no model can infer its way
  around a mis-weighted objective — which is exactly what the average-of-averages bug
  produces. It also makes the correct and buggy optima *computable by hand*.

In [3]:
PAD_ID, BOS_ID, EOS_ID = 0, 1, 2
BYTE0 = 3
VOCAB = 259


def encode(text):
    return [BOS_ID] + [BYTE0 + b for b in text.encode("utf-8")] + [EOS_ID]


def decode_id(i):
    if i == PAD_ID: return "<pad>"
    if i == BOS_ID: return "<bos>"
    if i == EOS_ID: return "<eos>"
    b = i - BYTE0
    ch = chr(b)
    return ch if 32 <= b < 127 else f"<{b:02x}>"


def decode(ids):
    return "".join(decode_id(i) for i in ids)


In [4]:
PROSE_DOCS = [
    "The loss is one number. The gradient attached to a weight is one number too: how "
    "much the loss would change if that single weight moved. A step reads a batch, "
    "computes the loss, fills in every gradient, and moves every weight at once.",
    "Backpropagation is the chain rule applied one link at a time, from the loss "
    "towards the inputs. The framework records every multiply and add during the "
    "forward pass; loss.backward() walks that record in reverse. Bookkeeping, not magic.",
    "Gradients accumulate. Calling backward twice without zeroing adds the second "
    "gradient onto the first, and no error is raised. Forgetting the wipe is a bug; "
    "doing it on purpose, micro-batch after micro-batch, is gradient accumulation.",
    "The batch size the optimizer sees is a choice, independent of what fits in "
    "memory. Run several small micro-batches, let the gradients pile up, step once, "
    "then wipe. The sum does not care how the parts were grouped.",
    "Normalize the loss by tokens, not by micro-batches. A short micro-batch must not "
    "get the same vote as a long one. Averages of averages are wrong whenever the "
    "counts differ, and they look right whenever the counts happen to match.",
    "A single anomalous batch can produce enormous gradients. Compute the combined "
    "norm of all gradients; if it exceeds the cap, scale every gradient by the same "
    "factor. Direction is preserved, only the length shrinks.",
    "The grad norm often moves before the loss does. A run heading toward failure can "
    "show it in the norm thousands of steps before the loss reflects it. Watch the "
    "norm; the loss curve is a lagging indicator.",
    "Training holds five things for every weight: the bf16 weight, its bf16 gradient, "
    "a full precision master copy, and two optimizer moments. Sixteen bytes per "
    "weight, before a single activation is stored.",
    "Exponent bits buy range, mantissa bits buy detail. bf16 keeps all eight exponent "
    "bits and pays with precision; fp16 made the opposite trade and loses small "
    "gradients to underflow unless the loss is scaled.",
    "The loss tells you whether the model is learning. MFU tells you whether the "
    "machine is being wasted. An eight percent run and a forty five percent run draw "
    "identical loss curves; one of them costs four times as much.",
]
TELEM_DOCS = [
    "loss 2.61 ok", "loss 2.48 ok", "loss 2.55 ok", "loss 2.39 ok",
    "loss 2.71 ok", "loss 2.44 ok", "loss 2.52 ok", "loss 2.36 ok",
]
HELD_PROSE = [
    "Print things and check things. Every serious training bug is silent, and the "
    "loss curve is not going to be the one that tells you. A plausible number is not "
    "evidence; a measured one, re-derived from disk, is.",
    "Choose the clip threshold from the observed distribution of grad norms, not "
    "from habit, and agree on an acceptable MFU before the run begins, so nobody has "
    "to negotiate it at three in the morning.",
]
HELD_TELEM = ["loss 2.58 ok", "loss 2.43 ok", "loss 2.66 ok"]

TRAIN_PROSE_IDS = []
for _doc in PROSE_DOCS:
    TRAIN_PROSE_IDS += encode(_doc)
HELD_PROSE_IDS = []
for _doc in HELD_PROSE:
    HELD_PROSE_IDS += encode(_doc)

A_ID = BYTE0 + ord("a")
HZ_SHORT_N, HZ_LONG_N = 2, 32          # 'a'-run lengths; the hazard sits after 2 a's


def hazard_doc(n, T):
    ids = [BOS_ID] + [A_ID] * n + [EOS_ID]
    return ids + [PAD_ID] * (T - len(ids))


In [5]:
n_prose = len(TRAIN_PROSE_IDS)
n_telem = sum(len(encode(t)) for t in TELEM_DOCS)
print(f"train prose stream : {n_prose:5d} tokens ({len(PROSE_DOCS)} docs)")
print(f"train telemetry    : {n_telem:5d} tokens ({len(TELEM_DOCS)} docs, "
      f"{min(len(encode(t)) for t in TELEM_DOCS)}-{max(len(encode(t)) for t in TELEM_DOCS)} tokens each)")
print(f"held-out prose     : {len(HELD_PROSE_IDS):5d} tokens | held-out telemetry: "
      f"{sum(len(encode(t)) for t in HELD_TELEM)} tokens")
print("sample telemetry doc:", decode(encode(TELEM_DOCS[0])))
print("sample hazard docs  :", decode(hazard_doc(2, 8)), "|",
      decode(hazard_doc(32, 36))[:40], "...")
RESULTS["corpus"] = {"train_prose_tokens": n_prose, "train_telem_tokens": n_telem}

train prose stream :  2203 tokens (10 docs)
train telemetry    :   112 tokens (8 docs, 14-14 tokens each)
held-out prose     :   408 tokens | held-out telemetry: 42 tokens
sample telemetry doc: <bos>loss 2.61 ok<eos>
sample hazard docs  : <bos>aa<eos><pad><pad><pad><pad> | <bos>aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa<eo ...


## §2 — The model, and the three loss-side helpers everything reuses

`TinyLM`: pre-norm transformer, untied head (unshared storage keeps §9's byte
accounting exact, and keeps §4's "gradient of this weight" unambiguous — a tied
parameter's gradient is the *sum* of its embedding-path and head-path contributions).
Attention is written out as explicit matmuls so §3 can show the score tensor rather
than hide it inside a fused kernel, and the causal mask is boolean so the same code
runs unchanged in float64 (§4 needs that).

Three helpers carry the whole assignment:
- `per_token_ce` — the shift and the mask in one place, returning *per-token* losses
  (the only honest currency for combining micro-batches). Valid counts are counted on
  the **shifted targets**: a document with L content bytes plus `<bos>`/`<eos>`
  contributes L+1 target slots, not L+2;
- `token_weighted` / `avg_of_avgs` — the correct and the broken combine;
- `global_grad_norm` — the single most useful trace on any training dashboard.

In [6]:
class Attention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dh = h, d // h
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.proj = nn.Linear(d, d, bias=False)

    def forward(self, x, trace=None, tag=""):
        B, T, d = x.shape
        qkv = self.qkv(x)                                   # (B, T, 3d)
        q, k, v = qkv.split(d, dim=2)
        q = q.view(B, T, self.h, self.dh).transpose(1, 2)   # (B, h, T, dh)
        k = k.view(B, T, self.h, self.dh).transpose(1, 2)
        v = v.view(B, T, self.h, self.dh).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.dh)   # (B, h, T, T)
        causal = torch.ones(T, T, dtype=torch.bool, device=x.device).triu(1)
        weights = torch.softmax(scores.masked_fill(causal, float("-inf")), dim=-1)
        ctx = weights @ v                                         # (B, h, T, dh)
        out = self.proj(ctx.transpose(1, 2).reshape(B, T, d))     # (B, T, d)
        if trace is not None:
            trace += [
                (f"{tag}.qkv", qkv, ("batch sequences", "positions", "3 stacked projections x model width")),
                (f"{tag}.q", q, ("batch sequences", "attention heads", "positions", "per-head width d/h")),
                (f"{tag}.scores", scores, ("batch sequences", "attention heads", "query position", "key position")),
                (f"{tag}.weights", weights, ("batch sequences", "attention heads", "query position", "key position (rows sum to 1)")),
                (f"{tag}.ctx", ctx, ("batch sequences", "attention heads", "positions", "per-head width d/h")),
                (f"{tag}.out", out, ("batch sequences", "positions", "model width, back in the residual stream")),
            ]
        return out


class MLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc1 = nn.Linear(d, 4 * d, bias=False)
        self.fc2 = nn.Linear(4 * d, d, bias=False)

    def forward(self, x, trace=None, tag=""):
        hidden = F.gelu(self.fc1(x))                        # (B, T, 4d)
        out = self.fc2(hidden)                              # (B, T, d)
        if trace is not None:
            trace += [
                (f"{tag}.hidden", hidden, ("batch sequences", "positions", "MLP expansion 4 x model width")),
                (f"{tag}.out", out, ("batch sequences", "positions", "model width")),
            ]
        return out


class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn, self.mlp = Attention(d, h), MLP(d)

    def forward(self, x, trace=None, tag=""):
        x = x + self.attn(self.ln1(x), trace, f"{tag}.attn")
        x = x + self.mlp(self.ln2(x), trace, f"{tag}.mlp")
        if trace is not None:
            trace.append((f"{tag}.resid", x,
                          ("batch sequences", "positions", "model width (residual stream)")))
        return x


class TinyLM(nn.Module):
    def __init__(self, d=CFG["d_model"], n_layer=CFG["n_layer"], n_head=CFG["n_head"],
                 vocab=VOCAB, max_pos=CFG["max_pos"]):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, d)
        self.pos_emb = nn.Embedding(max_pos, d)
        self.blocks = nn.ModuleList(Block(d, n_head) for _ in range(n_layer))
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab, bias=False)   # untied on purpose (SS2 text)
        for p in self.parameters():
            if p.dim() >= 2:
                nn.init.normal_(p, std=0.02)

    def forward(self, tokens, trace=None):
        B, T = tokens.shape
        te = self.tok_emb(tokens)                             # (B, T, d)
        pe = self.pos_emb(torch.arange(T, device=tokens.device))  # (T, d)
        x = te + pe
        if trace is not None:
            trace += [
                ("tokens", tokens, ("batch sequences", "positions (token ids)")),
                ("tok_emb", te, ("batch sequences", "positions", "model width")),
                ("pos_emb", pe, ("positions", "model width (broadcast over batch)")),
                ("x_embedded", x, ("batch sequences", "positions", "model width")),
            ]
        for li, blk in enumerate(self.blocks):
            x = blk(x, trace, f"block{li}")
        x = self.ln_f(x)
        logits = self.head(x)                                 # (B, T, V)
        if trace is not None:
            trace += [
                ("ln_f", x, ("batch sequences", "positions", "model width, normalized")),
                ("logits", logits, ("batch sequences", "positions", "one score per vocabulary id")),
            ]
        return logits


def per_token_ce(logits, tokens):
    """The shift and the mask in one place. Returns (per_tok, mask), both (B, T-1)."""
    shifted = logits[:, :-1]                    # position t predicts token t+1
    targets = tokens[:, 1:]
    per_tok = F.cross_entropy(
        shifted.reshape(-1, shifted.size(-1)), targets.reshape(-1),
        reduction="none").view(targets.shape)
    mask = (targets != PAD_ID).to(per_tok.dtype)
    return per_tok, mask


def token_weighted_loss(logits, tokens):
    per_tok, mask = per_token_ce(logits, tokens)
    return (per_tok * mask).sum() / mask.sum()


def token_weighted(means, counts):
    """Correct combine: every token carries equal weight."""
    return sum(m * c for m, c in zip(means, counts)) / sum(counts)


def avg_of_avgs(means, counts):
    """The pre-2024 framework bug: every micro-batch carries equal weight."""
    return sum(means) / len(means)


def global_grad_norm(model):
    sq = 0.0
    for p in model.parameters():
        if p.grad is not None:
            sq += float(p.grad.pow(2).sum())
    return math.sqrt(sq)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def prose_crop(rng, T=CFG["T"]):
    i = rng.randrange(0, len(TRAIN_PROSE_IDS) - T)
    return TRAIN_PROSE_IDS[i:i + T]


def eval_token_weighted(model, ids, T=CFG["T"]):
    """Full-stream sweep: consecutive windows, every target scored once,
    aggregated as global sum/sum (never a mean of batch means)."""
    rows = [ids[i:i + T] for i in range(0, len(ids) - T + 1, T - 1)]
    tail = len(ids) - (len(rows) * (T - 1) + 1)
    if tail > 1:
        rows.append(ids[-T:])          # last window overlaps; its overlap is masked out
        overlap = T - 1 - tail
    else:
        overlap = 0
    batch = torch.tensor(rows)
    with torch.no_grad():
        per_tok, mask = per_token_ce(model(batch), batch)
    if overlap:
        mask[-1, :overlap] = 0.0       # do not double count the overlapped targets
    return float((per_tok * mask).sum() / mask.sum())


In [7]:
seed_all()
model = TinyLM()
N_PARAMS = count_params(model)
N_EMB = model.tok_emb.weight.numel() + model.pos_emb.weight.numel()
print(f"TinyLM: {N_PARAMS:,} parameters ({N_EMB:,} in the two embedding tables)")
for name, p in model.named_parameters():
    print(f"  {name:24s} {str(tuple(p.shape)):14s} {p.numel():7,d}")
RESULTS["model"] = {"n_params": N_PARAMS, "n_emb": N_EMB}
HEADLINE["n_params"] = f"{N_PARAMS:,}"

TinyLM: 493,568 parameters (65,920 in the two embedding tables)
  tok_emb.weight           (259, 128)      33,152
  pos_emb.weight           (256, 128)      32,768
  blocks.0.ln1.weight      (128,)             128
  blocks.0.ln1.bias        (128,)             128
  blocks.0.ln2.weight      (128,)             128
  blocks.0.ln2.bias        (128,)             128
  blocks.0.attn.qkv.weight (384, 128)      49,152
  blocks.0.attn.proj.weight (128, 128)      16,384
  blocks.0.mlp.fc1.weight  (512, 128)      65,536
  blocks.0.mlp.fc2.weight  (128, 512)      65,536
  blocks.1.ln1.weight      (128,)             128
  blocks.1.ln1.bias        (128,)             128
  blocks.1.ln2.weight      (128,)             128
  blocks.1.ln2.bias        (128,)             128
  blocks.1.attn.qkv.weight (384, 128)      49,152
  blocks.1.attn.proj.weight (128, 128)      16,384
  blocks.1.mlp.fc1.weight  (512, 128)      65,536
  blocks.1.mlp.fc2.weight  (128, 512)      65,536
  ln_f.weight              (128,) 

## §3 — Task 1: every tensor in one step, every dimension explained

One complete step — forward, loss, backward, update, wipe — with every tensor that
occurs in it printed once: activations as the forward pass creates them, then every
parameter with its gradient (same shape: one number *per weight*), then what the
optimizer keeps between steps (Adam's two moments — the two bars under each weight in
the session's widget #4). The flattened views are printed too, because
`reshape(-1, V)` is where mis-sliced tensors go to hide.

In [8]:
seed_all()
step_model = TinyLM()
opt = torch.optim.AdamW(step_model.parameters(), lr=CFG["lr"])

rng = random.Random(SEED)
batch = torch.tensor([prose_crop(rng) for _ in range(CFG["B"])])

trace = []
logits = step_model(batch, trace=trace)

# ---- the loss side of the step, traced explicitly ----
shifted = logits[:, :-1]
targets = batch[:, 1:]
flat_logits = shifted.reshape(-1, VOCAB)
flat_targets = targets.reshape(-1)
per_tok = F.cross_entropy(flat_logits, flat_targets, reduction="none").view(targets.shape)
mask = (targets != PAD_ID).float()
loss = (per_tok * mask).sum() / mask.sum()
trace += [
    ("logits_shifted", shifted, ("batch sequences", "positions 0..T-2 (each predicts the NEXT token)", "one score per vocabulary id")),
    ("targets", targets, ("batch sequences", "positions 1..T-1 (the token each position demands)")),
    ("flat_logits", flat_logits, ("batch x positions flattened into rows", "one score per vocabulary id")),
    ("flat_targets", flat_targets, ("batch x positions flattened -- one demanded id per row",)),
    ("per_token_loss", per_tok, ("batch sequences", "positions -- one loss per predicted token")),
    ("mask", mask, ("batch sequences", "positions -- 1 real target, 0 padding")),
    ("loss", loss, ("scalar -- the one number the whole step exists to shrink",)),
]

W = max(len(n) for n, _, _ in trace)
print(f"{'-' * (W + 30)}\nFORWARD + LOSS -- every tensor, in creation order\n{'-' * (W + 30)}")
for name, t, dims in trace:
    shape = tuple(t.shape) if t.dim() else ()
    print(f"{name:{W}s} {str(shape):22s} {str(t.dtype).replace('torch.', ''):9s}")
    for ax, meaning in enumerate(dims):
        size = shape[ax] if ax < len(shape) else 1
        print(f"{'':{W}s}   dim{ax} = {size:<5d} {meaning}")
RESULTS["shapes"] = {n: list(t.shape) for n, t, _ in trace}
print(f"\n{len(trace)} tensors traced; loss = {fmt(float(loss))} "
      f"(ln V = {fmt(math.log(VOCAB))})")

-------------------------------------------------
FORWARD + LOSS -- every tensor, in creation order
-------------------------------------------------
tokens              (8, 128)               int64    
                      dim0 = 8     batch sequences
                      dim1 = 128   positions (token ids)
tok_emb             (8, 128, 128)          float32  
                      dim0 = 8     batch sequences
                      dim1 = 128   positions
                      dim2 = 128   model width
pos_emb             (128, 128)             float32  
                      dim0 = 128   positions
                      dim1 = 128   model width (broadcast over batch)
x_embedded          (8, 128, 128)          float32  
                      dim0 = 8     batch sequences
                      dim1 = 128   positions
                      dim2 = 128   model width
block0.attn.qkv     (8, 128, 384)          float32  
                      dim0 = 8     batch sequences
                      dim

/tmp/ipykernel_2653/1911603142.py:38: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:822.)
  print(f"\n{len(trace)} tensors traced; loss = {fmt(float(loss))} "


In [9]:
loss.backward()
print(f"{'-' * 78}\nBACKWARD -- one gradient per weight, so grad shape == weight shape"
      f"\n{'-' * 78}")
for name, p in step_model.named_parameters():
    print(f"{name:24s} weight {str(tuple(p.shape)):14s} grad {str(tuple(p.grad.shape)):14s}"
          f"  |grad| max {float(p.grad.abs().max()):.4f}")

opt.step()
print(f"\n{'-' * 78}\nOPTIMIZER STATE -- what Adam keeps between steps (per weight: two "
      f"running moments)\n{'-' * 78}")
n_state = 0
for name, p in step_model.named_parameters():
    st = opt.state[p]
    n_state += st["exp_avg"].numel() + st["exp_avg_sq"].numel()
    if name in ("tok_emb.weight", "blocks.0.attn.qkv.weight", "head.weight"):
        print(f"{name:24s} exp_avg {str(tuple(st['exp_avg'].shape)):14s} "
              f"exp_avg_sq {str(tuple(st['exp_avg_sq'].shape)):14s}")
print(f"... (same pattern for all {sum(1 for _ in step_model.parameters())} parameters: "
      f"{n_state:,} state values = 2 x {N_PARAMS:,} weights)")

opt.zero_grad()
assert all(p.grad is None or p.grad.abs().sum() == 0 for p in step_model.parameters())
print("\nzero_grad(): every gradient wiped -- the step is forward -> loss -> backward "
      "-> update -> WIPE.")

# audited shape relations
B, T, V, d = CFG["B"], CFG["T"], VOCAB, CFG["d_model"]
S = RESULTS["shapes"]
assert S["tokens"] == [B, T] and S["logits"] == [B, T, V]
assert S["logits_shifted"] == [B, T - 1, V] and S["targets"] == [B, T - 1]
assert S["flat_logits"] == [B * (T - 1), V] and S["flat_targets"] == [B * (T - 1)]
assert S["block0.attn.scores"] == [B, CFG["n_head"], T, T]
assert S["block0.attn.q"] == [B, CFG["n_head"], T, d // CFG["n_head"]]
assert S["block0.attn.qkv"] == [B, T, 3 * d] and S["block0.mlp.hidden"] == [B, T, 4 * d]
assert S["loss"] == []
RESULTS["shapes_step"] = {"n_traced": len(trace), "optimizer_state_values": n_state}
print("all shape relations asserted against the config.")

------------------------------------------------------------------------------
BACKWARD -- one gradient per weight, so grad shape == weight shape
------------------------------------------------------------------------------
tok_emb.weight           weight (259, 128)     grad (259, 128)      |grad| max 0.1860
pos_emb.weight           weight (256, 128)     grad (256, 128)      |grad| max 0.0790
blocks.0.ln1.weight      weight (128,)         grad (128,)          |grad| max 0.0081
blocks.0.ln1.bias        weight (128,)         grad (128,)          |grad| max 0.0276
blocks.0.ln2.weight      weight (128,)         grad (128,)          |grad| max 0.0081
blocks.0.ln2.bias        weight (128,)         grad (128,)          |grad| max 0.0157
blocks.0.attn.qkv.weight weight (384, 128)     grad (384, 128)      |grad| max 0.0690
blocks.0.attn.proj.weight weight (128, 128)     grad (128, 128)      |grad| max 0.0795
blocks.0.mlp.fc1.weight  weight (512, 128)     grad (512, 128)      |grad| max 0.0558


## §4 — Task 2: one gradient, verified by hand

### 4a. The session's toy chain first — and why the *central* nudge wins

`x=2, w1=3, w2=4, t=20`: forward gives h=6, y=24, loss=16. The chain rule, one link at
a time — ∂L/∂y = 8, ∂L/∂w2 = 48, ∂L/∂h = 32, **∂L/∂w1 = 64**. The session's forward
nudge (w1 → 3.001) reproduces its 16.064 → gradient ≈ 64.064 — and that 0.064 excess
is not noise, it is a *law*: for this loss the forward-difference error is **exactly
64·ε** (the second-derivative term, (w2·x)² = 64). The central difference cancels that
term, and because the loss is *quadratic* in w1 the cancellation is complete — central
is exact to float64 roundoff at every ε, so there is no ε-tuning story here at all.
(The U-curve lives in §4b, on the real model, where the loss is not a parabola.)
Autograd must land on the same 64 — that agreement *is* the proof that backprop = chain
rule, nothing more. All arithmetic in float64 on purpose; §4b shows what fp32 does to
this kind of check.

In [10]:
x, w1v, w2v, t = 2.0, 3.0, 4.0, 20.0   # python floats are float64

def toy_loss(w1_):
    return (w2v * w1_ * x - t) ** 2

# by hand, one link at a time
y = w2v * w1v * x
dL_dy = 2 * (y - t)            # 8
dL_dw2 = dL_dy * (w1v * x)     # 48
dL_dh = dL_dy * w2v            # 32
dL_dw1_hand = dL_dh * x        # 64

fwd_nudge = (toy_loss(3.001) - toy_loss(3.0)) / 0.001        # the session's 16.064
central = (toy_loss(3.001) - toy_loss(2.999)) / 0.002

w1 = torch.tensor(w1v, dtype=torch.float64, requires_grad=True)
(torch.tensor(w2v, dtype=torch.float64) * w1 * x - t).pow(2).backward()
autograd_w1 = float(w1.grad)

print(f"forward pass        : h={w1v * x:.0f}  y={y:.0f}  loss={toy_loss(3.0):.0f}")
print(f"nudged loss (3.001) : {toy_loss(3.001):.6f}   -> forward-diff grad {fwd_nudge:.6f}")
print(f"chain rule by hand  : dL/dy={dL_dy:.0f}  dL/dw2={dL_dw2:.0f}  dL/dh={dL_dh:.0f}  "
      f"dL/dw1={dL_dw1_hand:.0f}")
print(f"central difference  : {central:.12f}")
print(f"autograd            : {autograd_w1:.12f}")

# the error LAW of the forward nudge: err = 64 * eps, exactly, at every eps
print("\nforward-difference error is 64*eps exactly (the f''/2 term of the Taylor series):")
law = []
for k in range(2, 7):
    eps = 10.0 ** -k
    err = (toy_loss(w1v + eps) - toy_loss(w1v)) / eps - 64.0
    law.append({"eps": eps, "err": err, "err_over_eps": err / eps})
    print(f"  eps=1e-{k}: error {err:+.2e}  error/eps = {err / eps:.6f}")
assert abs(autograd_w1 - 64.0) < 1e-10 and abs(central - 64.0) < 1e-9
assert all(abs(z["err_over_eps"] - 64.0) < 0.64 for z in law)   # within 1%
RESULTS["toy_chain"] = {"hand": dL_dw1_hand, "fwd_nudge": fwd_nudge,
                        "central": central, "autograd": autograd_w1,
                        "nudged_loss": toy_loss(3.001), "fwd_error_law": law}
HEADLINE["toy_grad"] = fmt(autograd_w1, 4)

# cross-validation: torch's own numeric checker on the same function, float64
ok = torch.autograd.gradcheck(
    lambda w: (torch.tensor(w2v, dtype=torch.float64) * w * x - t) ** 2,
    (torch.tensor(w1v, dtype=torch.float64, requires_grad=True),))
print(f"\ntorch.autograd.gradcheck on the toy chain: {ok}")

forward pass        : h=6  y=24  loss=16
nudged loss (3.001) : 16.064064   -> forward-diff grad 64.064000
chain rule by hand  : dL/dy=8  dL/dw2=48  dL/dh=32  dL/dw1=64
central difference  : 63.999999999992
autograd            : 64.000000000000

forward-difference error is 64*eps exactly (the f''/2 term of the Taylor series):
  eps=1e-2: error +6.40e-01  error/eps = 64.000000
  eps=1e-3: error +6.40e-02  error/eps = 64.000000
  eps=1e-4: error +6.40e-03  error/eps = 64.000001
  eps=1e-5: error +6.40e-04  error/eps = 64.000047
  eps=1e-6: error +6.40e-05  error/eps = 64.007530

torch.autograd.gradcheck on the toy chain: True


### 4b. The real model: nudge one weight of half a million

The protocol, pinned before the numbers (all of it matters):

- **which weight**: the element `backward()` claims matters most (argmax |grad|) in
  `blocks.0.mlp.fc1.weight` — a deterministic rule, stated, on an untied tensor. A
  near-zero gradient would make the comparison vacuous — the sharpest version of that
  trap is real and shown below: embedding rows for byte ids *absent from the batch*
  have gradients that are exactly zero, and a finite difference on one of them
  "confirms" 0 = 0 while verifying nothing.
- **float64 clone** (`copy.deepcopy(model).double().eval()`): the live fp32 model and
  its optimizer are never touched; the bool causal mask means the same forward runs
  unchanged in double.
- **save/assign/restore**, never `+= eps` then `-= 2eps` (that leaves float residue);
  restoration is asserted bit-exact and the base loss is asserted to reproduce.
- **one thread** for this cell, so the numbers do not wobble with BLAS scheduling.
- **the ε sweep runs here**, where the loss is genuinely non-quadratic: truncation
  error falls as ε² on the right, cancellation noise rises as 1/ε on the left, and the
  plateau between them is the gradient. The same sweep on an fp32 clone is overlaid —
  including the ε where the fp32 loss difference is *exactly zero* (complete
  cancellation): the measuring stick drowns, not the gradient.

In [11]:
torch.set_num_threads(1)          # deterministic reductions for the check cell
fd_model = copy.deepcopy(model).double().eval()
fd_batch = batch[:4, :64]         # fixed, cached batch reused by every evaluation

def fd_loss(m):
    with torch.no_grad():
        return float(token_weighted_loss(m(fd_batch), fd_batch))

base_loss = fd_loss(fd_model)
for p in fd_model.parameters():
    p.grad = None
token_weighted_loss(fd_model(fd_batch), fd_batch).backward()

Wmat = fd_model.blocks[0].mlp.fc1.weight
flat_idx = int(Wmat.grad.abs().argmax())
r, c = divmod(flat_idx, Wmat.shape[1])
g_autograd = float(Wmat.grad[r, c])
orig = Wmat.data[r, c].clone()
print(f"chosen weight: blocks.0.mlp.fc1.weight[{r},{c}] = {float(orig):+.6f}")
print(f"autograd grad = {g_autograd:+.12e}   (base loss {base_loss:.12f})")

def central_diff(m, W_, r_, c_, orig_, eps):
    wp, wm = orig_ + eps, orig_ - eps
    W_.data[r_, c_] = wp; L_plus = fd_loss(m)
    W_.data[r_, c_] = wm; L_minus = fd_loss(m)
    W_.data[r_, c_] = orig_
    return (L_plus - L_minus) / float(wp - wm)   # realized step in the denominator

eps_grid = [10.0 ** -k for k in range(1, 11)]
fd_rows = []
for eps in eps_grid:
    g_fd = central_diff(fd_model, Wmat, r, c, orig, eps)
    abs_err = abs(g_fd - g_autograd)
    rel_err = abs_err / abs(g_autograd)
    fd_rows.append({"eps": eps, "g_fd": g_fd, "abs_err": abs_err, "rel_err": rel_err})
    print(f"eps=1e-{round(-math.log10(eps)):<2d} fd={g_fd:+.12e}  abs err {abs_err:.2e}"
          f"  rel err {rel_err:.2e}")

assert bool(torch.equal(Wmat.data[r, c], orig)), "restoration must be bit-exact"
assert fd_loss(fd_model) == base_loss, "base loss must reproduce after the sweep"

best = min(fd_rows, key=lambda z: z["rel_err"])
decimals = -math.log10(best["abs_err"]) if best["abs_err"] > 0 else 15.0
print(f"\nbest eps={best['eps']:.0e}: fd and backward() agree to rel err "
      f"{best['rel_err']:.2e}  (~{decimals:.1f} decimals of |gradient|~1e-2)")
assert best["rel_err"] < 1e-7, "float64 central difference must certify rel err < 1e-7"
RESULTS["finite_diff"] = {
    "weight": f"blocks.0.mlp.fc1.weight[{r},{c}]", "autograd": g_autograd,
    "base_loss": base_loss, "best_eps": best["eps"], "best_fd": best["g_fd"],
    "best_abs_err": best["abs_err"], "best_rel_err": best["rel_err"],
    "best_decimals": float(min(decimals, 15)), "sweep": fd_rows}
HEADLINE["fd_rel_err"] = f"{best['rel_err']:.1e}"
HEADLINE["fd_decimals"] = f"{min(decimals, 15):.1f}"

chosen weight: blocks.0.mlp.fc1.weight[347,104] = +0.020560
autograd grad = +5.447958539893e-02   (base loss 5.563863544232)
eps=1e-1  fd=+5.421610554731e-02  abs err 2.63e-04  rel err 4.84e-03
eps=1e-2  fd=+5.447695742538e-02  abs err 2.63e-06  rel err 4.82e-05
eps=1e-3  fd=+5.447955911952e-02  abs err 2.63e-08  rel err 4.82e-07


eps=1e-4  fd=+5.447958514093e-02  abs err 2.58e-10  rel err 4.74e-09
eps=1e-5  fd=+5.447958537630e-02  abs err 2.26e-11  rel err 4.15e-10


eps=1e-6  fd=+5.447958484334e-02  abs err 5.56e-10  rel err 1.02e-08


eps=1e-7  fd=+5.447958972870e-02  abs err 4.33e-09  rel err 7.95e-08
eps=1e-8  fd=+5.447953198765e-02  abs err 5.34e-08  rel err 9.80e-07
eps=1e-9  fd=+5.447997611466e-02  abs err 3.91e-07  rel err 7.17e-06


eps=1e-10 fd=+5.448530442905e-02  abs err 5.72e-06  rel err 1.05e-04

best eps=1e-05: fd and backward() agree to rel err 4.15e-10  (~10.6 decimals of |gradient|~1e-2)


In [12]:
# one element per parameter TYPE, same fp64 protocol, one good eps each --
# the whole gradient is then certified in one shot by a directional derivative.
eps0 = 1e-5


def central_diff_flat(m, W_, fi, eps):
    """Central difference on element fi of W_ (any rank), save/assign/restore."""
    flat = W_.data.view(-1)
    orig_ = flat[fi].clone()
    flat[fi] = orig_ + eps; L_plus = fd_loss(m)
    flat[fi] = orig_ - eps; L_minus = fd_loss(m)
    flat[fi] = orig_
    return (L_plus - L_minus) / (2 * eps)


in_batch_id = int(fd_batch.flatten()[0])
picks = [("tok_emb.weight (in-batch row)", fd_model.tok_emb.weight,
          in_batch_id * CFG["d_model"]
          + int(fd_model.tok_emb.weight.grad[in_batch_id].abs().argmax())),
         ("blocks.0.attn.qkv.weight", fd_model.blocks[0].attn.qkv.weight, None),
         ("blocks.0.ln1.weight", fd_model.blocks[0].ln1.weight, None),
         ("blocks.1.mlp.fc2.weight", fd_model.blocks[1].mlp.fc2.weight, None),
         ("head.weight", fd_model.head.weight, None)]
type_rows = []
for label, W_, fi in picks:
    fi = int(W_.grad.abs().argmax()) if fi is None else fi
    ga = float(W_.grad.view(-1)[fi])
    gfd = central_diff_flat(fd_model, W_, fi, eps0)
    rel = abs(gfd - ga) / max(abs(ga), 1e-30)
    type_rows.append({"param": label, "flat_idx": fi, "autograd": ga, "fd": gfd,
                      "rel_err": rel})
    print(f"{label:34s} autograd {ga:+.6e}  fd {gfd:+.6e}  rel err {rel:.1e}")
assert all(z["rel_err"] < 1e-5 for z in type_rows)

# the zero-gradient trap, shown instead of stepped on
used = set(fd_batch.flatten().tolist())
unused = next(i for i in range(VOCAB) if i not in used)
g_row = fd_model.tok_emb.weight.grad[unused]
print(f"\ntrap: token id {unused} ('{decode_id(unused)}') never occurs in the batch -> "
      f"its embedding row grad is exactly {float(g_row.abs().sum()):.1f}")
print("a finite difference there would 'confirm' 0 == 0 while verifying nothing; "
      "the selection rule (argmax |grad|) exists to dodge this.")
n_zero_rows = int((fd_model.tok_emb.weight.grad.abs().sum(1) == 0).sum())
print(f"({n_zero_rows} of {VOCAB} embedding rows have exactly-zero gradient on this batch)")

# directional derivative: ONE scalar that checks EVERY gradient at once
gen = torch.Generator().manual_seed(SEED)
vecs = [torch.randn(p.shape, generator=gen, dtype=torch.float64)
        for p in fd_model.parameters()]
vnorm = math.sqrt(sum(float(v.pow(2).sum()) for v in vecs))
vecs = [v / vnorm for v in vecs]
g_dot_v = sum(float((p.grad * v).sum()) for p, v in zip(fd_model.parameters(), vecs))
eps_d = 1e-6
with torch.no_grad():
    for p, v in zip(fd_model.parameters(), vecs): p.add_(eps_d * v)
    Lp = fd_loss(fd_model)
    for p, v in zip(fd_model.parameters(), vecs): p.add_(-2 * eps_d * v)
    Lm = fd_loss(fd_model)
    for p, v in zip(fd_model.parameters(), vecs): p.add_(eps_d * v)
fd_dir = (Lp - Lm) / (2 * eps_d)
rel_dir = abs(fd_dir - g_dot_v) / abs(g_dot_v)
print(f"\ndirectional derivative over ALL {N_PARAMS:,} weights: "
      f"fd {fd_dir:+.10e} vs g.v {g_dot_v:+.10e}  rel err {rel_dir:.1e}")
assert rel_dir < 1e-6
RESULTS["finite_diff"]["per_type"] = type_rows
RESULTS["finite_diff"]["zero_grad_rows"] = n_zero_rows
RESULTS["finite_diff"]["directional"] = {"fd": fd_dir, "g_dot_v": g_dot_v,
                                         "rel_err": rel_dir}

tok_emb.weight (in-batch row)      autograd +8.843607e-02  fd +8.843607e-02  rel err 2.4e-10
blocks.0.attn.qkv.weight           autograd +7.824116e-02  fd +7.824116e-02  rel err 7.5e-10


blocks.0.ln1.weight                autograd -8.624632e-03  fd -8.624632e-03  rel err 2.7e-10
blocks.1.mlp.fc2.weight            autograd -3.017886e-02  fd -3.017886e-02  rel err 6.9e-11


head.weight                        autograd +2.245989e-01  fd +2.245989e-01  rel err 1.9e-10

trap: token id 0 ('<pad>') never occurs in the batch -> its embedding row grad is exactly 0.0
a finite difference there would 'confirm' 0 == 0 while verifying nothing; the selection rule (argmax |grad|) exists to dodge this.
(224 of 259 embedding rows have exactly-zero gradient on this batch)



directional derivative over ALL 493,568 weights: fd +1.4734597542e-02 vs g.v +1.4734597351e-02  rel err 1.3e-08


In [13]:
# the same sweep on an fp32 clone: the probe drowns, the gradient does not
fd32 = copy.deepcopy(model).eval()   # fp32
for p in fd32.parameters():
    p.grad = None
token_weighted_loss(fd32(fd_batch), fd_batch).backward()
W32 = fd32.blocks[0].mlp.fc1.weight
g32 = float(W32.grad[r, c])
orig32 = W32.data[r, c].clone()

def fd32_loss():
    with torch.no_grad():
        return float(token_weighted_loss(fd32(fd_batch), fd_batch))

fd32_rows = []
for eps in eps_grid:
    W32.data[r, c] = orig32 + eps; Lp = fd32_loss()
    W32.data[r, c] = orig32 - eps; Lm = fd32_loss()
    W32.data[r, c] = orig32
    g_fd = (Lp - Lm) / (2 * eps)
    fd32_rows.append({"eps": eps, "g_fd": g_fd,
                      "abs_err": abs(g_fd - g32),
                      "zero_diff": Lp == Lm})
best32 = min(z["abs_err"] / abs(g32) for z in fd32_rows)
n_dead = sum(z["zero_diff"] for z in fd32_rows)
grad_vs_fp64 = abs(g32 - g_autograd) / abs(g_autograd)
print(f"fp32 probe, best rel err : {best32:.1e}   "
      f"({n_dead} eps values give L(w+eps) == L(w-eps) EXACTLY -- total cancellation)")
print(f"fp32 backward() itself   : rel diff vs the fp64 gradient {grad_vs_fp64:.1e}")
print("=> the precision limit is in the measuring stick, not the gradient.")
RESULTS["finite_diff"]["fp32"] = {"best_rel_err": best32, "n_cancelled": n_dead,
                                  "grad_rel_diff_vs_fp64": grad_vs_fp64,
                                  "sweep": fd32_rows}
assert best32 < 1e-2 and grad_vs_fp64 < 1e-4

fig, axis = plt.subplots(figsize=(7, 4.2))
axis.loglog([z["eps"] for z in fd_rows], [max(z["abs_err"], 1e-16) for z in fd_rows],
            "o-", label="float64 probe")
axis.loglog([z["eps"] for z in fd32_rows],
            [max(z["abs_err"], 1e-16) for z in fd32_rows],
            "s--", label="float32 probe (drowns; 0-diff points pinned at 1e-16)")
axis.axhline(abs(g_autograd), color="gray", lw=0.7, ls=":",
             label="|gradient| itself")
axis.set_xlabel("nudge size eps"); axis.set_ylabel("|finite difference - backward()|")
axis.set_title("The nudge, done live: truncation (left) vs cancellation (right)")
axis.invert_xaxis(); axis.grid(alpha=0.3); axis.legend(fontsize=8)
fig.tight_layout(); fig.savefig(ART / "plots" / "fd_ucurve.png", dpi=120)
plt.close(fig)
torch.set_num_threads(N_THREADS)
print("saved plots/fd_ucurve.png; threads restored to", N_THREADS)

fp32 probe, best rel err : 4.6e-04   (5 eps values give L(w+eps) == L(w-eps) EXACTLY -- total cancellation)
fp32 backward() itself   : rel diff vs the fp64 gradient 3.9e-07
=> the precision limit is in the measuring stick, not the gradient.


saved plots/fd_ucurve.png; threads restored to 4


## §5 — Task 3: gradient accumulation, broken on purpose

### 5a. The mechanism first: gradients pile up, and the session's arithmetic, realized

Two `backward()` calls without a wipe **add** — asserted bitwise, not `allclose`. That
"bug" is the mechanism accumulation exploits on purpose.

Then the session's exact numbers (micro-batches averaging 2.0, 2.0, 5.0 over 4, 4, 2
tokens), realized as *actual logits* whose per-token cross-entropy is exactly those
values — not hardcoded arithmetic — so the artifact carries per-token losses the audit
can re-combine both ways from disk:

- token-weighted (correct): (4·2.0 + 4·2.0 + 2·5.0) / 10 = **2.6**
- average of averages (the pre-2024 framework bug): (2.0 + 2.0 + 5.0) / 3 = **3.0**
- error: 0.4 / 2.6 = **15.4%** — and it vanishes whenever the counts happen to match,
  which is why casual testing never caught it. (This 15.4% is a *measurement* error on
  one contrived batch; the training-curve effect in §5c is a different, separately
  measured number — the two must not be conflated.)

In [14]:
# gradients ACCUMULATE: two backwards, no wipe, bitwise-equal to 2x one backward
seed_all()
acc_model = TinyLM(d=64, n_layer=1, n_head=2)
tb = torch.tensor([prose_crop(random.Random(0), 64)])
p0 = acc_model.blocks[0].mlp.fc1.weight

acc_model.zero_grad(set_to_none=True)
print(f"before any backward : p.grad is {p0.grad}")
token_weighted_loss(acc_model(tb), tb).backward()
g1 = p0.grad.clone()
print(f"after 1st backward  : p.grad.abs().sum() = {float(g1.abs().sum()):.6f}")
token_weighted_loss(acc_model(tb), tb).backward()      # NO zero_grad in between
assert torch.equal(p0.grad, 2 * g1)                    # bitwise, not allclose
print(f"after 2nd backward  : p.grad == 2 x first, bitwise ({float(p0.grad.abs().sum()):.6f})")
print("(zero_grad(set_to_none=True) resets to None, not zeros -- the default since 1.7)")

before any backward : p.grad is None
after 1st backward  : p.grad.abs().sum() = 75.091324
after 2nd backward  : p.grad == 2 x first, bitwise (150.182648)
(zero_grad(set_to_none=True) resets to None, not zeros -- the default since 1.7)


In [15]:
def logits_with_ce(target_ids, ce_value, vocab=VOCAB):
    """Construct one row of logits per target whose CE against the target is exactly
    ce_value: p(target) = exp(-ce), the rest of the mass uniform on the other ids."""
    p_t = math.exp(-ce_value)
    other = (1 - p_t) / (vocab - 1)
    rows = torch.full((len(target_ids), vocab), math.log(other))
    for i, t in enumerate(target_ids):
        rows[i, t] = math.log(p_t)
    return rows

micro_defs = [([10, 11, 12, 13], 2.0),      # 4 valid tokens, mean loss 2.0
              ([14, 15, 16, 17], 2.0),      # 4 valid tokens, mean loss 2.0
              ([18, 19], 5.0)]              # 2 valid tokens, mean loss 5.0
per_token_dump, means, counts = [], [], []
for ids_, ce in micro_defs:
    lg = logits_with_ce(ids_, ce)
    pt = F.cross_entropy(lg, torch.tensor(ids_), reduction="none")
    per_token_dump.append([float(v) for v in pt])
    means.append(float(pt.mean())); counts.append(len(ids_))
    print(f"micro-batch: {len(ids_)} tokens, per-token CE {[f'{v:.4f}' for v in pt]}")

correct = token_weighted(means, counts)
buggy = avg_of_avgs(means, counts)
err_pct = abs(buggy - correct) / correct * 100
print(f"\ntoken-weighted (correct)   : {fmt(correct)}")
print(f"average of averages (buggy): {fmt(buggy)}")
print(f"error                      : {fmt(err_pct, 1)}% -- and 0% whenever counts match")
assert abs(correct - 2.6) < 1e-6 and abs(buggy - 3.0) < 1e-6
eq = avg_of_avgs([2.0, 3.0, 4.0], [5, 5, 5]) - token_weighted([2.0, 3.0, 4.0], [5, 5, 5])
assert abs(eq) < 1e-12   # equal counts: the bug is invisible
RESULTS["static_mirror"] = {"per_token": per_token_dump, "means": means,
                            "counts": counts, "correct": correct, "buggy": buggy,
                            "err_pct": err_pct}
HEADLINE["mirror_correct"], HEADLINE["mirror_buggy"] = fmt(correct), fmt(buggy)
HEADLINE["mirror_err"] = fmt(err_pct, 1)

micro-batch: 4 tokens, per-token CE ['2.0000', '2.0000', '2.0000', '2.0000']
micro-batch: 4 tokens, per-token CE ['2.0000', '2.0000', '2.0000', '2.0000']
micro-batch: 2 tokens, per-token CE ['5.0000', '5.0000']

token-weighted (correct)   : 2.6000
average of averages (buggy): 3.0000
error                      : 15.4% -- and 0% whenever counts match


### 5b. The identity that makes accumulation legitimate — and the one the bug breaks

Correct accumulation scales each micro-batch's *summed* loss by 1/(total tokens) and
lets the gradients pile up. That must equal one big batch, exactly — verified here on
four micro-batches of genuinely different lengths (3, 3, 3, 33 target slots): forward
logits at valid positions first (localizes any failure to forward vs backward), then
every parameter's gradient, in fp32 and again on a float64 copy where the identity is
machine-clean. The buggy combine is run through the identical harness and lands far
away — same code path, different denominator. Dropout would break the identity (each
forward draws a different mask); this model has none, which is also the V4 postmortem's
reversibility rule.

In [16]:
def accum_grads(m, micros, mode):
    """Accumulated gradients over micro-batches under either combine rule."""
    m.zero_grad(set_to_none=True)
    counts = [int((torch.tensor(mb)[:, 1:] != PAD_ID).sum()) for mb in micros]
    total = sum(counts)
    for mb in micros:
        tb = torch.tensor(mb)
        pt, msk = per_token_ce(m(tb), tb)
        if mode == "correct":
            ((pt * msk).sum() / total).backward()
        else:                                  # average of averages
            ((pt * msk).sum() / msk.sum() / len(micros)).backward()
    return [p.grad.clone() for p in m.parameters()], counts


def rel_l2(gs_a, gs_b):
    num = math.sqrt(sum(float((a - b).pow(2).sum()) for a, b in zip(gs_a, gs_b)))
    den = math.sqrt(sum(float(b.pow(2).sum()) for b in gs_b))
    return num / den


In [17]:
seed_all()
id_model = TinyLM(d=CFG["hz_d"], n_layer=2, n_head=2)
micros_id = [[hazard_doc(2, CFG["hz_T"])], [hazard_doc(2, CFG["hz_T"])],
             [hazard_doc(2, CFG["hz_T"])], [hazard_doc(32, CFG["hz_T"])]]

# reference: ONE big batch (the 4 rows stacked), token-weighted
big = torch.tensor([mb[0] for mb in micros_id])
id_model.zero_grad(set_to_none=True)
pt, msk = per_token_ce(id_model(big), big)
((pt * msk).sum() / msk.sum()).backward()
g_big = [p.grad.clone() for p in id_model.parameters()]

# forward identity at valid positions: each row alone == the same row in the big batch
with torch.no_grad():
    lg_big = id_model(big)
    max_fwd_diff = max(float((id_model(torch.tensor(mb)) - lg_big[i:i + 1]).abs().max())
                       for i, mb in enumerate(micros_id))
print(f"forward identity: micro logits vs big-batch logits, max |diff| = {max_fwd_diff:.2e}")

g_acc, counts = accum_grads(id_model, micros_id, "correct")
g_bug, _ = accum_grads(id_model, micros_id, "buggy")
err_ok = rel_l2(g_acc, g_big)
err_bug = rel_l2(g_bug, g_big)
print(f"target counts per micro-batch: {counts} (counted on the SHIFTED targets)")
print(f"correct accumulation vs big batch: rel L2 err {err_ok:.2e}   <- the identity")
print(f"buggy accumulation   vs big batch: rel L2 err {err_bug:.2e}   <- the bug")

# the same identity on a float64 copy: machine-clean
id64 = copy.deepcopy(id_model).double()
id64.zero_grad(set_to_none=True)
pt, msk = per_token_ce(id64(big), big)
((pt * msk).sum() / msk.sum()).backward()
g_big64 = [p.grad.clone() for p in id64.parameters()]
g_acc64, _ = accum_grads(id64, micros_id, "correct")
err64 = rel_l2(g_acc64, g_big64)
print(f"float64 copy                     : rel L2 err {err64:.2e}")
assert max_fwd_diff < 1e-5 and err_ok < 1e-5 and err64 < 1e-12 and err_bug > 1e-2
RESULTS["accum_identity"] = {"counts": counts, "fwd_max_diff": max_fwd_diff,
                             "rel_err_correct": err_ok, "rel_err_correct_f64": err64,
                             "rel_err_buggy": err_bug}
HEADLINE["acc_id_f64"] = f"{err64:.1e}"
HEADLINE["acc_id_bug"] = f"{err_bug:.1e}"

forward identity: micro logits vs big-batch logits, max |diff| = 0.00e+00
target counts per micro-batch: [3, 3, 3, 33] (counted on the SHIFTED targets)
correct accumulation vs big batch: rel L2 err 9.43e-08   <- the identity
buggy accumulation   vs big batch: rel L2 err 5.67e-01   <- the bug
float64 copy                     : rel L2 err 1.61e-16


### 5c. The two curves the assignment asks for — on a corpus where the bug must show

**Why not just train on prose?** Because a transformer *infers its way around* a
mis-weighted mixture: given context, it learns each register conditionally, and the
reweighting barely moves the optimum (§5d measures exactly that, and reports it as the
honest negative). To make the bug express itself structurally, the conflicted decision
must sit at positions whose prefixes are *identical* across document types. The hazard
corpus does that: documents are `<bos>aa<eos>` (3 target slots) or `<bos>` + 32 `a`s +
`<eos>` (33 slots), 50/50, **one document per micro-batch** — the composition under
which per-micro-batch averaging actually happens in the wild (per-sequence loss
averaging in SFT fine-tuning). After two `a`s the document ends or continues with true
probability ½, and no context can tell which.

Both optima are then computable by hand, before any training:

- **correct arm** (every token votes once): p(eos | `<bos>aa`) → **1/2**;
  best possible token-weighted eval = 2·ln2 / 36 = **ln 2 / 18 ≈ 0.0385**;
- **buggy arm** (every *document* votes once): the short document's hazard slot gets
  1/3 of a vote, the long one's 1/33, so p → (1/3)/(1/3 + 1/33) = **11/12 ≈ 0.917**;
  its token-weighted eval tends to (−ln(11/12) − ln(1/12))/36 ≈ **0.0715**.

The buggy model comes to believe ~92% of documents end after two bytes when the truth
is 50% — a silently substituted objective, not noisy training. Both arms run from
identical inits on identical document streams (checksummed), no clipping, AdamW; three
seeds; evaluated on a common held-out set with the *correct* token-weighted protocol
(global sum/sum — the eval must not itself commit the bug under study), plus the
doc-weighted eval, where the sign **flips**: the buggy arm wins the metric that matches
its objective. A trained probe reads out p(eos | `<bos>aa`) directly.

One deliberate control in the stream itself: every step holds exactly **two short and
two long documents** (order shuffled), so the per-step token total is a constant 72.
That is not cosmetic — §5c″ shows what happens when it varies.

In [18]:
LN2 = math.log(2.0)
HZ_ANALYTIC = {
    "correct_tw": LN2 / 18,
    "buggy_p": 11 / 12,
    "buggy_tw": (-math.log(11 / 12) - math.log(1 / 12)) / 36,
    "correct_dw": (LN2 / 3 + LN2 / 33) / 2,
    "buggy_dw": (-math.log(11 / 12) / 3 + -math.log(1 / 12) / 33) / 2,
}


def hazard_stream(seed, steps, K):
    """Fixed composition: 2 short + 2 long per step (order shuffled) -> the per-step
    token total is constant (72), so normalizing by it is unbiased. SS5c'' shows the
    subtle bias that appears when the composition is random instead."""
    assert K == 4
    rng = random.Random(seed)
    out = []
    for _ in range(steps):
        docs = [hazard_doc(HZ_SHORT_N, CFG["hz_T"]), hazard_doc(HZ_SHORT_N, CFG["hz_T"]),
                hazard_doc(HZ_LONG_N, CFG["hz_T"]), hazard_doc(HZ_LONG_N, CFG["hz_T"])]
        rng.shuffle(docs)
        out.append([[d_] for d_ in docs])
    return out


def random_hazard_stream(seed, steps, K):
    """The naive stream: each micro-batch's document drawn 50/50 -> the per-step
    token total VARIES, and SS5c'' measures what that alone does."""
    rng = random.Random(seed)
    return [[[hazard_doc(HZ_SHORT_N if rng.random() < 0.5 else HZ_LONG_N,
                         CFG["hz_T"])] for _ in range(K)] for _ in range(steps)]


def hazard_eval_set(seed=990, n=64):
    rng = random.Random(seed)
    return [hazard_doc(HZ_SHORT_N if rng.random() < 0.5 else HZ_LONG_N, CFG["hz_T"])
            for _ in range(n)]


def eval_hazard(m, docs):
    """token-weighted (global sum/sum), doc-weighted (mean of per-doc means),
    and the probe p(eos | <bos> a a)."""
    batch = torch.tensor(docs)
    with torch.no_grad():
        pt, msk = per_token_ce(m(batch), batch)
        probs = torch.softmax(m(torch.tensor([[BOS_ID, A_ID, A_ID]]))[0, -1], dim=-1)
    tw = float((pt * msk).sum() / msk.sum())
    dw = float(((pt * msk).sum(1) / msk.sum(1)).mean())
    return tw, dw, float(probs[EOS_ID]), (pt * msk)


def train_hazard(mode, seed, steps=None, log=None):
    steps = steps or CFG["steps_accum"]
    torch.manual_seed(seed)
    m = TinyLM(d=CFG["hz_d"], n_layer=2, n_head=2)
    init_sha = sha([p.sum().item() for p in m.parameters()])
    opt = torch.optim.AdamW(m.parameters(), lr=CFG["lr"])
    stream = hazard_stream(seed + 100, steps, CFG["accum_K"])
    ev = hazard_eval_set()
    curve = {"step": [], "tw": [], "dw": [], "probe": []}
    for s, micros in enumerate(stream):
        counts = [int((torch.tensor(mb)[:, 1:] != PAD_ID).sum()) for mb in micros]
        total = sum(counts)
        opt.zero_grad(set_to_none=True)
        for mb in micros:
            tb = torch.tensor(mb)
            pt, msk = per_token_ce(m(tb), tb)
            if mode == "correct":
                ((pt * msk).sum() / total).backward()
            else:
                ((pt * msk).sum() / msk.sum() / len(micros)).backward()
        opt.step()                      # NO clipping anywhere in this section
        if s % CFG["eval_every"] == 0 or s == steps - 1:
            tw, dw, probe, _ = eval_hazard(m, ev)
            curve["step"].append(s); curve["tw"].append(tw)
            curve["dw"].append(dw); curve["probe"].append(probe)
    return m, curve, init_sha, sha(stream)


In [19]:
print("analytic, before any training:")
print(f"  correct arm: p(eos|aa) -> 0.5     token-weighted floor {HZ_ANALYTIC['correct_tw']:.4f}")
print(f"  buggy arm  : p(eos|aa) -> {HZ_ANALYTIC['buggy_p']:.4f}  token-weighted asymptote "
      f"{HZ_ANALYTIC['buggy_tw']:.4f}")
print(f"  doc-weighted flips: correct {HZ_ANALYTIC['correct_dw']:.4f} vs buggy "
      f"{HZ_ANALYTIC['buggy_dw']:.4f}\n")

hz_runs = {}
for seed in CFG["accum_seeds"]:
    for mode in ("correct", "buggy"):
        t0 = time.time()
        m_, curve, ish, dsh = train_hazard(mode, seed)
        hz_runs[(seed, mode)] = {"curve": curve, "init_sha": ish, "stream_sha": dsh,
                                 "model": m_}
        lastk = curve["tw"][-5:]
        print(f"seed {seed} {mode:7s}: final-5 token-weighted eval "
              f"{sum(lastk) / len(lastk):.4f}  probe p(eos|aa)={curve['probe'][-1]:.3f}  "
              f"doc-weighted {curve['dw'][-1]:.4f}  [{time.time() - t0:.0f}s]")
    a, b = hz_runs[(seed, "correct")], hz_runs[(seed, "buggy")]
    assert a["init_sha"] == b["init_sha"] and a["stream_sha"] == b["stream_sha"], \
        "arms must share init and data bitwise"

def last5(xs):
    return sum(xs[-5:]) / len(xs[-5:])

summary = []
for seed in CFG["accum_seeds"]:
    c5 = hz_runs[(seed, "correct")]["curve"]["tw"][-5:]
    b5 = hz_runs[(seed, "buggy")]["curve"]["tw"][-5:]
    summary.append({"seed": seed,
                    "correct_tw": sum(c5) / len(c5), "buggy_tw": sum(b5) / len(b5),
                    "gap": sum(b5) / len(b5) - sum(c5) / len(c5),
                    "correct_probe": last5(hz_runs[(seed, "correct")]["curve"]["probe"]),
                    "buggy_probe": last5(hz_runs[(seed, "buggy")]["curve"]["probe"]),
                    "correct_dw": hz_runs[(seed, "correct")]["curve"]["dw"][-1],
                    "buggy_dw": hz_runs[(seed, "buggy")]["curve"]["dw"][-1]})
gaps = [z["gap"] for z in summary]
print(f"\ngap (buggy - correct), token-weighted, mean of last 5 evals: "
      f"{[f'{g:.4f}' for g in gaps]}")
print(f"doc-weighted flips in every seed: "
      f"{all(z['buggy_dw'] < z['correct_dw'] for z in summary)}")
for z in summary:
    strict_assert(z["gap"] > 0.015, "the gap must be unmistakable in every seed")
    strict_assert(0.35 <= z["correct_probe"] <= 0.65, "correct arm must learn p ~ 0.5")
    strict_assert(z["buggy_probe"] > 0.80, "buggy arm must learn the substituted objective")
    strict_assert(z["buggy_dw"] < z["correct_dw"], "buggy must WIN its own (doc-weighted) metric")
RESULTS["accum_training"] = {"analytic": HZ_ANALYTIC, "seeds": summary}
CURVES["hazard"] = {f"{seed}_{mode}": hz_runs[(seed, mode)]["curve"]
                    for seed in CFG["accum_seeds"] for mode in ("correct", "buggy")}
s0 = summary[0]
HEADLINE["hz_correct"], HEADLINE["hz_buggy"] = fmt(s0["correct_tw"]), fmt(s0["buggy_tw"])
HEADLINE["hz_gap"] = fmt(s0["gap"])
HEADLINE["hz_probe_c"], HEADLINE["hz_probe_b"] = fmt(s0["correct_probe"], 3), fmt(s0["buggy_probe"], 3)

analytic, before any training:
  correct arm: p(eos|aa) -> 0.5     token-weighted floor 0.0385
  buggy arm  : p(eos|aa) -> 0.9167  token-weighted asymptote 0.0714
  doc-weighted flips: correct 0.1260 vs buggy 0.0522



seed 1337 correct: final-5 token-weighted eval 0.0424  probe p(eos|aa)=0.499  doc-weighted 0.1366  [8s]


seed 1337 buggy  : final-5 token-weighted eval 0.0714  probe p(eos|aa)=0.916  doc-weighted 0.0506  [8s]


seed 1338 correct: final-5 token-weighted eval 0.0449  probe p(eos|aa)=0.295  doc-weighted 0.2282  [8s]


seed 1338 buggy  : final-5 token-weighted eval 0.0715  probe p(eos|aa)=0.916  doc-weighted 0.0508  [8s]


seed 1339 correct: final-5 token-weighted eval 0.0425  probe p(eos|aa)=0.499  doc-weighted 0.1367  [9s]


seed 1339 buggy  : final-5 token-weighted eval 0.0715  probe p(eos|aa)=0.916  doc-weighted 0.0507  [8s]

gap (buggy - correct), token-weighted, mean of last 5 evals: ['0.0290', '0.0267', '0.0290']
doc-weighted flips in every seed: True


In [20]:
seed0 = CFG["accum_seeds"][0]
cC = hz_runs[(seed0, "correct")]["curve"]; cB = hz_runs[(seed0, "buggy")]["curve"]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
ax = axes[0]
ax.plot(cC["step"], cC["tw"], label="correct (token-weighted combine)", color="tab:blue")
ax.plot(cB["step"], cB["tw"], label="buggy (average of averages)", color="tab:red")
ax.axhline(HZ_ANALYTIC["correct_tw"], ls=":", color="tab:blue", lw=1,
           label=f"analytic floor {HZ_ANALYTIC['correct_tw']:.4f}")
ax.axhline(HZ_ANALYTIC["buggy_tw"], ls=":", color="tab:red", lw=1,
           label=f"analytic asymptote {HZ_ANALYTIC['buggy_tw']:.4f}")
ax.set_xlabel("step"); ax.set_ylabel("held-out token-weighted CE (nats)")
ax.set_title(f"Task 3: both curves, together (seed {seed0})")
ax.set_yscale("log"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
ax = axes[1]
for z in RESULTS["accum_training"]["seeds"]:
    s = z["seed"]
    d = [b - c for b, c in zip(CURVES["hazard"][f"{s}_buggy"]["tw"],
                               CURVES["hazard"][f"{s}_correct"]["tw"])]
    ax.plot(CURVES["hazard"][f"{s}_correct"]["step"], d, lw=1, label=f"seed {s}")
ax.axhline(HZ_ANALYTIC["buggy_tw"] - HZ_ANALYTIC["correct_tw"], ls=":", color="k",
           label="analytic gap")
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("step"); ax.set_ylabel("buggy - correct (nats)")
ax.set_title("the gap, all seeds"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
ax = axes[2]
ax.plot(cC["step"], cC["probe"], color="tab:blue", label="correct arm")
ax.plot(cB["step"], cB["probe"], color="tab:red", label="buggy arm")
ax.axhline(0.5, ls=":", color="tab:blue", lw=1, label="truth 0.5")
ax.axhline(11 / 12, ls=":", color="tab:red", lw=1, label="buggy optimum 11/12")
ax.set_xlabel("step"); ax.set_ylabel("model's p(eos | <bos>aa)")
ax.set_title("what each arm comes to believe"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(ART / "plots" / "accum_gap.png", dpi=120)
plt.close(fig); print("saved plots/accum_gap.png")

# per-token eval losses of the final seed-0 models, for the independent audit
_, _, _, ptC = eval_hazard(hz_runs[(seed0, "correct")]["model"], hazard_eval_set())
_, _, _, ptB = eval_hazard(hz_runs[(seed0, "buggy")]["model"], hazard_eval_set())
RESULTS["accum_training"]["per_token_final"] = {
    "correct": [[float(v) for v in row] for row in ptC],
    "buggy": [[float(v) for v in row] for row in ptB],
    "mask_counts": [int((torch.tensor(d)[1:] != PAD_ID).sum()) for d in hazard_eval_set()]}

saved plots/accum_gap.png


### 5c″. A subtler cousin, found while building this section: the ratio bias

Why did §5c pin the per-step composition to 2+2? Because with a *random* 50/50 draw,
even the "correct" arm converges visibly off 0.5 — this notebook's first version
measured p(eos|aa) ≈ 0.67 and had to explain it. The reason is worth the detour:
normalizing each step by **that step's own token count** makes the objective
E[(1/N_step)·Σ ce], not E[Σ ce]/E[N_step] — a *ratio* estimator. A short document's
hazard slot tends to appear in steps with smaller N (its own document contributed only
3 tokens), so `eos`-labeled slots systematically carry more weight than `a`-labeled
ones. The bias is exactly computable — enumerate the 8 compositions of the other three
micro-batches:

p* = E[1/N | short] / (E[1/N | short] + E[1/N | long]) = **0.6681**

and it is measured below. The sharp version of "normalize by tokens": divide by a
**constant** (the expected or planned tokens per batch), not by whatever this batch
happened to contain, whenever the count varies with the content. At real batch sizes N
concentrates and the bias shrinks — at K = 4 with heavy-tailed lengths it is enormous.

In [21]:
from itertools import product
num = den = Fraction(0)
for others in product([HZ_SHORT_N + 1, HZ_LONG_N + 1], repeat=CFG["accum_K"] - 1):
    num += Fraction(1, 8) * Fraction(1, HZ_SHORT_N + 1 + sum(others))
    den += Fraction(1, 8) * Fraction(1, HZ_LONG_N + 1 + sum(others))
P_STAR = float(num / (num + den))
print(f"analytic ratio-bias optimum: p* = {P_STAR:.4f} (not 0.5!)")

rb_probes = []
rb_steps = CFG["steps_accum"] + (0 if FAST else 200)
for seed in CFG["accum_seeds"][:2]:
    torch.manual_seed(seed)
    m_ = TinyLM(d=CFG["hz_d"], n_layer=2, n_head=2)
    opt_ = torch.optim.AdamW(m_.parameters(), lr=CFG["lr"])
    probes_ = []
    ev_ = hazard_eval_set()
    for s, micros in enumerate(random_hazard_stream(seed + 100, rb_steps, CFG["accum_K"])):
        total = sum(int((torch.tensor(mb)[:, 1:] != PAD_ID).sum()) for mb in micros)
        opt_.zero_grad(set_to_none=True)
        for mb in micros:
            tb = torch.tensor(mb)
            pt, msk = per_token_ce(m_(tb), tb)
            ((pt * msk).sum() / total).backward()    # the "correct" combine, per-step N
        opt_.step()
        if s % CFG["eval_every"] == 0 or s == rb_steps - 1:
            probes_.append(eval_hazard(m_, ev_)[2])
    p_meas = sum(probes_[-10:]) / len(probes_[-10:])
    rb_probes.append(p_meas)
    print(f"seed {seed}: measured p(eos|aa) = {p_meas:.3f}  (p* = {P_STAR:.4f}, truth 0.5)")
strict_assert(all(abs(p_ - P_STAR) < 0.15 and p_ > 0.55 for p_ in rb_probes),
              "the ratio bias must pull the probe visibly toward p*, away from 0.5")
RESULTS["accum_ratio_bias"] = {"analytic_p": P_STAR, "measured_probes": rb_probes,
                               "steps": rb_steps}
HEADLINE["ratio_p_star"] = fmt(P_STAR)

analytic ratio-bias optimum: p* = 0.6681 (not 0.5!)


seed 1337: measured p(eos|aa) = 0.638  (p* = 0.6681, truth 0.5)


seed 1338: measured p(eos|aa) = 0.740  (p* = 0.6681, truth 0.5)


### 5c′. The negative control: equal token counts, and the bug cannot express itself

Same corpus, same code, one change: micro-batches are built to hold **exactly 33 target
slots each** (eleven short documents, or one long one). With equal counts, 1/K *is*
n/N — the two combine rules compute the same number by different arithmetic. Both arms
run; the gradient identity is asserted at step 0, and the curves stay within float
drift of each other. This isolates unequal counts as the sole cause of §5c's gap.

In [22]:
def control_stream(seed, steps, K=CFG["accum_K"]):
    rng = random.Random(seed)
    out = []
    for _ in range(steps):
        out.append([[hazard_doc(HZ_SHORT_N, HZ_SHORT_N + 3) for _ in range(11)]
                    if rng.random() < 0.5 else [hazard_doc(HZ_LONG_N, CFG["hz_T"])]
                    for _k in range(K)])
    return out


def train_control(mode, seed, steps=None):
    steps = steps or max(100, CFG["steps_accum"] // 2)
    torch.manual_seed(seed)
    m = TinyLM(d=CFG["hz_d"], n_layer=2, n_head=2)
    opt = torch.optim.AdamW(m.parameters(), lr=CFG["lr"])
    stream = control_stream(seed + 100, steps)
    ev = hazard_eval_set()
    curve = []
    for s, micros in enumerate(stream):
        counts = [int((torch.tensor(mb)[:, 1:] != PAD_ID).sum()) for mb in micros]
        assert len(set(counts)) == 1, "control requires equal counts"
        total = sum(counts)
        opt.zero_grad(set_to_none=True)
        for mb in micros:
            tb = torch.tensor(mb)
            pt, msk = per_token_ce(m(tb), tb)
            if mode == "correct":
                ((pt * msk).sum() / total).backward()
            else:
                ((pt * msk).sum() / msk.sum() / len(micros)).backward()
        opt.step()
        if s % CFG["eval_every"] == 0 or s == steps - 1:
            curve.append(eval_hazard(m, ev)[0])
    return m, curve

# step-0 gradient identity under equal counts
torch.manual_seed(seed0)
ctrl_m = TinyLM(d=CFG["hz_d"], n_layer=2, n_head=2)
ctrl_micros = control_stream(seed0 + 100, 1)[0]
gc_, _ = accum_grads(ctrl_m, ctrl_micros, "correct")
gb_, _ = accum_grads(ctrl_m, ctrl_micros, "buggy")
ctrl_grad_err = rel_l2(gb_, gc_)
print(f"equal counts, step-0 gradients: buggy vs correct rel L2 err {ctrl_grad_err:.2e}")

_, curveC = train_control("correct", seed0)
_, curveB = train_control("buggy", seed0)
ctrl_max_diff = max(abs(a - b) for a, b in zip(curveC, curveB))
hz_gap = RESULTS["accum_training"]["seeds"][0]["gap"]
print(f"equal-count control: max |eval diff| over training = {ctrl_max_diff:.5f}")
print(f"(the unequal-count gap was {hz_gap:.4f} -> "
      f"{hz_gap / max(ctrl_max_diff, 1e-9):.0f}x larger)")
assert ctrl_grad_err < 1e-5
strict_assert(ctrl_max_diff < 0.25 * hz_gap, "control must isolate unequal counts")
RESULTS["accum_control"] = {"grad_rel_err": ctrl_grad_err,
                            "max_curve_diff": ctrl_max_diff, "hazard_gap": hz_gap}

equal counts, step-0 gradients: buggy vs correct rel L2 err 0.00e+00


equal-count control: max |eval diff| over training = 0.00000
(the unequal-count gap was 0.0290 -> 28974997x larger)


### 5d. The honest negative: on ordinary mixed text, the same bug barely shows

The intuitive construction — long prose documents plus short telemetry lines, a
"different register" that average-of-averages overweights ~7× — was this notebook's
first design for §5c. Run for real, the gap is barely visible: the model infers the
register from the first tokens and fits both *conditionally*, so re-weighting the
mixture hardly moves the optimum. It is reported here, measured, instead of silently
replaced: the bug's damage depends on *what the overweighted tokens disagree about*,
not on the weighting alone. (It also shows the bug's stealth: on realistic data both
curves look fine — which is why it survived in every major framework until 2024.)

In [23]:
def mixed_stream(seed, steps, K=CFG["accum_K"]):
    rng = random.Random(seed)
    out = []
    for _ in range(steps):
        micros = [[prose_crop(rng)] for _ in range(K - 1)]
        tel = encode(TELEM_DOCS[rng.randrange(len(TELEM_DOCS))])
        micros.append([tel + [PAD_ID] * (CFG["T"] - len(tel))])
        rng.shuffle(micros)
        out.append(micros)
    return out


def eval_mixed(m):
    prose_tw = eval_token_weighted(m, HELD_PROSE_IDS)
    tel_rows = []
    for t_ in HELD_TELEM:
        ids_ = encode(t_)
        tel_rows.append(ids_ + [PAD_ID] * (CFG["T"] - len(ids_)))
    tb = torch.tensor(tel_rows)
    with torch.no_grad():
        pt, msk = per_token_ce(m(tb), tb)
    tel_tw = float((pt * msk).sum() / msk.sum())
    # mixture weights = the training stream's true token shares
    w_tel = MIX_TELEM_SHARE
    return prose_tw * (1 - w_tel) + tel_tw * w_tel, prose_tw, tel_tw


def train_mixed(mode, seed, steps=None):
    steps = steps or CFG["steps_accum"]
    torch.manual_seed(seed)
    m = TinyLM()
    opt = torch.optim.AdamW(m.parameters(), lr=CFG["lr"])
    stream = mixed_stream(seed + 100, steps)
    curve = {"step": [], "mix": [], "prose": [], "telem": []}
    for s, micros in enumerate(stream):
        counts = [int((torch.tensor(mb)[:, 1:] != PAD_ID).sum()) for mb in micros]
        total = sum(counts)
        opt.zero_grad(set_to_none=True)
        for mb in micros:
            tb = torch.tensor(mb)
            pt, msk = per_token_ce(m(tb), tb)
            if mode == "correct":
                ((pt * msk).sum() / total).backward()
            else:
                ((pt * msk).sum() / msk.sum() / len(micros)).backward()
        opt.step()
        if s % CFG["eval_every"] == 0 or s == steps - 1:
            mix, pr, te = eval_mixed(m)
            curve["step"].append(s); curve["mix"].append(mix)
            curve["prose"].append(pr); curve["telem"].append(te)
    return curve

# token shares of the two registers in the training stream (exact, from composition)
probe_stream = mixed_stream(seed0 + 100, 200)
tel_toks = pros_toks = 0
for micros in probe_stream:
    for mb in micros:
        n_ = int((torch.tensor(mb)[:, 1:] != PAD_ID).sum())
        if mb[0][0] == BOS_ID and PAD_ID in mb[0]:
            tel_toks += n_
        else:
            pros_toks += n_
MIX_TELEM_SHARE = tel_toks / (tel_toks + pros_toks)
print(f"telemetry token share of the stream: {MIX_TELEM_SHARE:.4f} "
      f"(the buggy combine hands it 1/{CFG['accum_K']} = 0.25 -> "
      f"{0.25 / MIX_TELEM_SHARE:.1f}x overweight)")

mixC = train_mixed("correct", seed0)
mixB = train_mixed("buggy", seed0)
mix_gap = (sum(mixB["mix"][-5:]) - sum(mixC["mix"][-5:])) / 5
mix_level = sum(mixC["mix"][-5:]) / 5
prose_gap = (sum(mixB["prose"][-5:]) - sum(mixC["prose"][-5:])) / 5
telem_gap = (sum(mixB["telem"][-5:]) - sum(mixC["telem"][-5:])) / 5
rel_mix = mix_gap / mix_level
hz_level = RESULTS["accum_training"]["seeds"][0]["correct_tw"]
rel_hz = hz_gap / hz_level
print(f"mixed-eval gap (buggy - correct): {mix_gap:+.4f} nats on a "
      f"{mix_level:.3f}-nat loss = {rel_mix * 100:+.1f}% relative  "
      f"[prose {prose_gap:+.4f}, telemetry {telem_gap:+.4f}]")
print(f"hazard-corpus gap for comparison: {hz_gap:+.4f} on {hz_level:.4f} "
      f"= {rel_hz * 100:+.0f}% relative")
print(f"-> the SAME bug is ~{rel_hz / max(abs(rel_mix), 1e-9):.0f}x less visible on "
      f"ordinary mixed text: two curves {abs(rel_mix) * 100:.1f}% apart look like one "
      f"line on a training chart, which is exactly how this bug survived until 2024")
RESULTS["accum_mixed_negative"] = {
    "telem_share": MIX_TELEM_SHARE, "mix_gap": mix_gap, "mix_level": mix_level,
    "rel_gap": rel_mix, "hazard_rel_gap": rel_hz,
    "prose_gap": prose_gap, "telem_gap": telem_gap}
CURVES["mixed"] = {"correct": mixC, "buggy": mixB}

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(mixC["step"], mixC["mix"], color="tab:blue", label="correct")
ax.plot(mixB["step"], mixB["mix"], color="tab:red", label="buggy")
ax.set_xlabel("step"); ax.set_ylabel("held-out mixed CE (nats)")
ax.set_title("SS5d honest negative: on ordinary mixed text the curves nearly coincide")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(ART / "plots" / "accum_negative.png", dpi=120)
plt.close(fig); print("saved plots/accum_negative.png")

telemetry token share of the stream: 0.0330 (the buggy combine hands it 1/4 = 0.25 -> 7.6x overweight)


mixed-eval gap (buggy - correct): +0.0418 nats on a 2.668-nat loss = +1.6% relative  [prose +0.0431, telemetry +0.0018]
hazard-corpus gap for comparison: +0.0290 on 0.0424 = +68% relative
-> the SAME bug is ~44x less visible on ordinary mixed text: two curves 1.6% apart look like one line on a training chart, which is exactly how this bug survived until 2024
saved plots/accum_negative.png


## §6 — Task 4: the grad norm at every step, and the step where it spoke first

**What is honestly claimable here, and what is not.** The per-step training loss and
the grad norm come from the *same* forward/backward on the *same* batch — when an
anomalous batch arrives at step k they spike **together**, and any claim that the raw
train loss lagged is bookkeeping theater. What actually lags is the *model*: the damage
done by the bad update can only appear in losses computed on *clean* data *after* the
update — and it persists (momentum keeps re-applying a share of the bad gradient for
~1/(1−β₁) steps) long after the one-step blip has scrolled off the loss chart. And the
smoothed loss every real dashboard draws lags further still.

So this section logs **three series** per arm, every step: the train loss on the
consumed batch, the pre-clip global grad norm, and a clean probe (full-stream sweep of
held-out prose, after the update — position stated). A 300-step CPU run cannot wait for
an organic incident, so one is planted, and framed as exactly that: **instrument
validation on a known event**. The anomalous batch is a shard decoded with the wrong
encoding — bytes uniform over the whole byte range — drawn from a dedicated generator
so every arm sees the identical clean stream (checksummed). (A gentler corruption, the
corpus with its byte *order* shuffled, is also run and reported: incidents come in
sizes.) The arms, differing *only* in the studied intervention:

| arm | injection | clipping |
|---|---|---|
| A placebo | no | no |
| B incident | step k | no |
| C policy, tight cap | step k | **every step**, cap = 2× the median of A's settled norms (from data, not habit — and A's pre-k steps are bitwise B's) |
| C2 policy, loose cap | step k | every step, cap just above the *largest* norm A ever showed in settled operation — a pure incident guard |
| D oracle | step k | at step k only — a labeled *counterfactual* (no real run knows k), isolating what the cap did at the moment it mattered |

Two caps on purpose: how often a cap binds on *ordinary* steps decides what it is. A
tight cap that binds nearly every step is really a gradient normalizer (an implicit LR
policy — measured below, including the fact that it *helps* here); a loose cap that
binds only on the anomaly is the session's safety valve. Both are reported, with their
binding counts.

SGD with momentum is used *deliberately*: its update is proportional to the gradient,
so the mechanism clipping guards is undistorted. AdamW's per-coordinate normalization
absorbs most of a one-step spike — measured at the end of the section as the honest
note on what the modern default buys you before clipping even enters.

In [24]:
def norm_probe_loss(m):
    return eval_token_weighted(m, HELD_PROSE_IDS)


def clean_batches(seed, steps):
    rng = random.Random(seed)
    return [[prose_crop(rng) for _ in range(CFG["B"])] for _ in range(steps)]


def anomalous_batch(seed, kind="wrong_encoding"):
    """The anomalous batch, from a DEDICATED generator (the clean stream's RNG is
    never touched, so every arm sees identical clean batches)."""
    g = torch.Generator().manual_seed(seed)
    if kind == "wrong_encoding":          # bytes uniform over the whole byte range
        return torch.randint(BYTE0, VOCAB, (CFG["B"], CFG["T"]), generator=g)
    content = [i for i in TRAIN_PROSE_IDS if i >= BYTE0]   # "shuffled": order destroyed
    idx = torch.randperm(len(content), generator=g)[:CFG["B"] * CFG["T"]]
    return torch.tensor([[content[int(j)] for j in idx[b * CFG["T"]:(b + 1) * CFG["T"]]]
                         for b in range(CFG["B"])])


def run_norm_arm(inject, clip_mode, cap=None, seed=SEED, steps=None, k=None,
                 opt_kind="sgd", lr=None, anomaly="wrong_encoding"):
    steps, k = steps or CFG["steps_norm"], k or CFG["inj_step"]
    torch.manual_seed(seed)
    m = TinyLM()
    lr = lr if lr is not None else (SGD_LR if opt_kind == "sgd" else CFG["lr"])
    opt = (torch.optim.SGD(m.parameters(), lr=lr, momentum=0.9) if opt_kind == "sgd"
           else torch.optim.AdamW(m.parameters(), lr=lr))
    stream = clean_batches(seed + 7, steps)
    anom = anomalous_batch(4242, anomaly)
    out = {"train": [], "norm": [], "probe": [], "clipped_at": [],
           "stream_sha": sha(stream)}
    for s in range(steps):
        batch = anom if (inject and s == k) else torch.tensor(stream[s])
        opt.zero_grad(set_to_none=True)
        pt, msk = per_token_ce(m(batch), batch)
        ((pt * msk).sum() / msk.sum()).backward()
        do_clip = clip_mode == "always" or (clip_mode == "oracle" and s == k)
        norm = float(torch.nn.utils.clip_grad_norm_(
            m.parameters(), cap if do_clip else float("inf"),
            error_if_nonfinite=True))
        if do_clip and norm > cap:
            out["clipped_at"].append(s)
        opt.step()
        out["train"].append(float((pt * msk).sum() / msk.sum()))
        out["norm"].append(norm)
        out["probe"].append(norm_probe_loss(m))   # AFTER the update, every step
    return out


def trailing_median(xs, t, w=30):
    lo = max(0, t - w)
    return sorted(xs[lo:t])[len(xs[lo:t]) // 2]


def detect_norm_spike(norms, burn_in=40, factor=5.0):
    for t_ in range(burn_in, len(norms)):
        if norms[t_] > factor * trailing_median(norms, t_):
            return t_
    return None


def detect_probe_damage(probes, burn_in=40, margin=0.02, consec=2):
    run = 0
    for t_ in range(burn_in, len(probes)):
        if probes[t_] > trailing_median(probes, t_) + margin:
            run += 1
            if run >= consec:
                return t_ - consec + 1
        else:
            run = 0
    return None


In [25]:
SGD_LR = 0.2   # pinned; the placebo arm below must be verifiably stable at this LR
               # (0.3 was tested and diverges before the injection ever arrives)
K_INJ = CFG["inj_step"]

t0 = time.time()
armA = run_norm_arm(inject=False, clip_mode="none")
armB = run_norm_arm(inject=True, clip_mode="none")
assert armA["stream_sha"] == armB["stream_sha"]
# both caps, chosen from the DATA (the placebo's settled norms, steps 40..k)
settled = armA["norm"][40:K_INJ]
CAP = 2.0 * sorted(settled)[len(settled) // 2]       # tight: 2 x median
CAP_LOOSE = 1.25 * max(settled)                      # loose: above all normal operation
armC = run_norm_arm(inject=True, clip_mode="always", cap=CAP)
armC2 = run_norm_arm(inject=True, clip_mode="always", cap=CAP_LOOSE)
armD = run_norm_arm(inject=True, clip_mode="oracle", cap=CAP)
armE = run_norm_arm(inject=True, clip_mode="none", opt_kind="adamw")
armB2 = run_norm_arm(inject=True, clip_mode="none", anomaly="shuffled")
print(f"7 arms x {CFG['steps_norm']} steps in {time.time() - t0:.0f}s | "
      f"tight cap 2 x median = {CAP:.3f} | loose cap 1.25 x max = {CAP_LOOSE:.3f}")

# placebo stability + pre-k bitwise equality between A and B
strict_assert(armA["train"][-1] < armA["train"][5], "placebo must be learning")
assert all(a == b for a, b in zip(armA["norm"][:K_INJ], armB["norm"][:K_INJ])), \
    "A and B must be bitwise identical before the injection"
assert all(math.isfinite(v) for v in armA["norm"] + armA["probe"] +
           armB["norm"] + armB["probe"] + armC["norm"] + armC["probe"])

7 arms x 300 steps in 70s | tight cap 2 x median = 0.365 | loose cap 1.25 x max = 1.096


In [26]:
med_pre = trailing_median(armB["norm"], K_INJ)
spike_B = detect_norm_spike(armB["norm"])
spike_A = detect_norm_spike(armA["norm"])
dmg_B = detect_probe_damage(armB["probe"])
dmg_A = detect_probe_damage(armA["probe"])

pre_probe = armB["probe"][K_INJ - 1]
worst_B = max(armB["probe"][K_INJ:K_INJ + 40])
worst_C = max(armC["probe"][K_INJ:K_INJ + 40])
worst_D = max(armD["probe"][K_INJ:K_INJ + 40])
worst_E = max(armE["probe"][K_INJ:K_INJ + 40])
worst_B2 = max(armB2["probe"][K_INJ:K_INJ + 40])
preB2 = armB2["probe"][K_INJ - 1]
worst_C2 = max(armC2["probe"][K_INJ:K_INJ + 40])
preC2 = armC2["probe"][K_INJ - 1]
preC = armC["probe"][K_INJ - 1]; preD = armD["probe"][K_INJ - 1]
preE = armE["probe"][K_INJ - 1]
rec_B = next((j - K_INJ for j in range(K_INJ, len(armB["probe"]))
              if armB["probe"][j] <= pre_probe + 0.01), None)
scale_at_k = min(1.0, CAP / armB["norm"][K_INJ])

print(f"norm at step {K_INJ}: {armB['norm'][K_INJ]:.2f} vs settled median {med_pre:.3f} "
      f"-> {armB['norm'][K_INJ] / med_pre:.1f}x")
print(f"train loss at k: {armB['train'][K_INJ]:.3f} (batch's own loss; spikes WITH the norm)")
print(f"detected: norm spike at step {spike_B} (placebo: {spike_A}); "
      f"probe damage from step {dmg_B} (placebo: {dmg_A})")
print(f"probe: pre-k {pre_probe:.4f} -> worst {worst_B:.4f} (+{worst_B - pre_probe:.4f}), "
      f"recovered after {rec_B} steps")
print(f"clipped arm C (tight): worst +{worst_C - preC:.4f} | C2 (loose): "
      f"+{worst_C2 - preC2:.4f} | oracle D: +{worst_D - preD:.4f} | "
      f"AdamW unclipped E: +{worst_E - preE:.4f}")
print(f"gentler incident (shuffled corpus bytes, unclipped): norm ratio "
      f"{armB2['norm'][K_INJ] / med_pre:.1f}x, damage +{worst_B2 - preB2:.4f} "
      f"-- incidents come in sizes; the norm ranks them before the probe can")
print(f"clip scale factor at k: tight cap {CAP:.3f} / norm {armB['norm'][K_INJ]:.2f} "
      f"= x{scale_at_k:.3f}")
def bind_counts(arm):
    early = len([s for s in arm["clipped_at"] if s < 40])
    settled = len([s for s in arm["clipped_at"] if 40 <= s < K_INJ])
    post = len([s for s in arm["clipped_at"] if s > K_INJ])
    return early, settled, post

eC, bindC, pC_ = bind_counts(armC)
eC2, bindC2, pC2_ = bind_counts(armC2)
n_settled = K_INJ - 40
print(f"cap binding in SETTLED operation (steps 40..{K_INJ - 1}): "
      f"tight {bindC}/{n_settled} (a gradient normalizer wearing a guard's uniform -- "
      f"and it helps: C's pre-incident probe {preC:.3f} beats B's {pre_probe:.3f}), "
      f"loose {bindC2}/{n_settled} (a pure incident guard)")
print(f"(both caps also clip the init transient -- tight {eC}, loose {eC2} of the "
      f"first 40 steps: step-0 gradients are every run's first anomaly -- and the "
      f"loose cap clips {pC2_} damaged steps after k)")

# what the dashboard's smoothed loss shows at k, vs what the norm shows
ema, ema_series = None, []
for v in armB["train"]:
    ema = v if ema is None else 0.98 * ema + 0.02 * v
    ema_series.append(ema)
ema_move = ema_series[K_INJ] - ema_series[K_INJ - 1]
ema_rel = ema_move / ema_series[K_INJ - 1]
print(f"at k the norm reads {armB['norm'][K_INJ] / med_pre:.1f}x its median; the "
      f"dashboard's EMA loss moves {ema_move:+.3f} nats ({ema_rel * 100:+.1f}% -- a "
      f"wiggle), and even that wiggle is the anomalous batch's own loss, not yet the "
      f"model's damage")

strict_assert(spike_B == K_INJ and spike_A is None, "spike detector must localize k")
strict_assert(dmg_B is not None and K_INJ <= dmg_B <= K_INJ + 3 and dmg_A is None,
              "probe damage must follow the update, placebo silent")
strict_assert(worst_B - pre_probe > 0.05, "the unclipped damage must be unmistakable")
strict_assert(worst_C - preC < 0.5 * (worst_B - pre_probe), "the tight cap must contain most of it")
strict_assert(worst_C2 - preC2 < 0.5 * (worst_B - pre_probe), "the loose cap must too")

# the session's arithmetic, checked: norm 8.4 against cap 1.0
assert abs(min(1.0, 1.0 / 8.4) - 0.119) < 5e-4
RESULTS["gradnorm"] = {
    "sgd_lr": SGD_LR, "k": K_INJ, "cap": CAP, "cap_loose": CAP_LOOSE,
    "median_norm": med_pre,
    "norm_at_k": armB["norm"][K_INJ], "spike_step": spike_B,
    "damage_step": dmg_B, "placebo_spike": spike_A, "placebo_damage": dmg_A,
    "probe_pre": pre_probe, "probe_worst_B": worst_B, "probe_worst_C": worst_C,
    "probe_worst_C2": worst_C2, "probe_worst_D": worst_D, "probe_worst_E": worst_E,
    "probe_pre_C": preC, "probe_pre_C2": preC2, "probe_pre_D": preD,
    "probe_pre_E": preE,
    "recovery_steps_B": rec_B, "scale_at_k": scale_at_k,
    "cap_binds_settled": bindC, "cap_loose_binds_settled": bindC2,
    "cap_binds_early": eC, "cap_loose_binds_early": eC2,
    "cap_loose_binds_post": pC2_, "n_settled": n_settled,
    "ema_move_at_k": ema_move, "ema_rel_at_k": ema_rel,
    "session_check_scale": min(1.0, 1.0 / 8.4),
    "shuffled_norm_ratio": armB2["norm"][K_INJ] / med_pre,
    "shuffled_damage": worst_B2 - preB2}
CURVES["gradnorm"] = {arm: {kk: vv for kk, vv in d.items() if kk != "stream_sha"}
                      for arm, d in
                      [("A", armA), ("B", armB), ("C", armC), ("C2", armC2),
                       ("D", armD), ("E", armE), ("B2", armB2)]}
HEADLINE["norm_ratio"] = fmt(armB["norm"][K_INJ] / med_pre, 1)
HEADLINE["dmg_B"] = fmt(worst_B - pre_probe)
HEADLINE["dmg_C"] = fmt(worst_C - preC)
HEADLINE["scale_at_k"] = fmt(scale_at_k, 3)

norm at step 150: 3.07 vs settled median 0.188 -> 16.3x
train loss at k: 9.547 (batch's own loss; spikes WITH the norm)
detected: norm spike at step 150 (placebo: None); probe damage from step 150 (placebo: None)
probe: pre-k 3.0427 -> worst 3.6970 (+0.6543), recovered after 17 steps
clipped arm C (tight): worst +0.1162 | C2 (loose): +0.1071 | oracle D: +0.0176 | AdamW unclipped E: +0.1459
gentler incident (shuffled corpus bytes, unclipped): norm ratio 1.1x, damage +0.0065 -- incidents come in sizes; the norm ranks them before the probe can
clip scale factor at k: tight cap 0.365 / norm 3.07 = x0.119
cap binding in SETTLED operation (steps 40..149): tight 110/110 (a gradient normalizer wearing a guard's uniform -- and it helps: C's pre-incident probe 2.879 beats B's 3.043), loose 4/110 (a pure incident guard)
(both caps also clip the init transient -- tight 40, loose 37 of the first 40 steps: step-0 gradients are every run's first anomaly -- and the loose cap clips 18 damaged steps aft

In [27]:
steps_x = list(range(CFG["steps_norm"]))
fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True)
ax = axes[0]
ax.semilogy(steps_x, armB["norm"], color="tab:red", lw=1, label="grad norm (pre-clip)")
ax.semilogy(steps_x, armA["norm"], color="tab:gray", lw=0.8, alpha=0.6, label="placebo")
ax.axhline(CAP, color="k", ls=":", lw=1, label=f"cap {CAP:.2f} (2x settled median)")
ax.axvline(K_INJ, color="k", lw=0.5, alpha=0.5)
ax.set_ylabel("global grad norm"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_title("the alarm: known BEFORE the update is applied")
ax = axes[1]
ax.plot(steps_x, armB["train"], color="tab:orange", lw=0.8, label="train loss (own batch)")
ax.plot(steps_x, ema_series, color="tab:brown", lw=1.5,
        label=f"EMA(0.98): moves {ema_move:+.2f} at k -- a wiggle")
ax.axvline(K_INJ, color="k", lw=0.5, alpha=0.5)
ax.set_ylabel("train CE"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_title("what the dashboard draws: the blip at k is the batch, not the model")
ax = axes[2]
ax.plot(steps_x, armB["probe"], color="tab:red", lw=1.2, label="B: unclipped")
ax.plot(steps_x, armC["probe"], color="tab:green", lw=1.2, label="C: tight cap, every step")
ax.plot(steps_x, armC2["probe"], color="tab:cyan", lw=1, label="C2: loose cap (incident guard)")
ax.plot(steps_x, armD["probe"], color="tab:olive", lw=1, ls="--",
        label="D: oracle clip at k (counterfactual)")
ax.plot(steps_x, armA["probe"], color="tab:gray", lw=0.8, alpha=0.6, label="A: placebo")
ax.axvline(K_INJ, color="k", lw=0.5, alpha=0.5)
ax.set_ylabel("clean probe CE (post-update)"); ax.set_xlabel("step")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_title("the damage: only visible on clean data, only after the update, and it lingers")
fig.tight_layout(); fig.savefig(ART / "plots" / "gradnorm_lead.png", dpi=120)
plt.close(fig); print("saved plots/gradnorm_lead.png")
print(f"\nthe honest sentence: at step {K_INJ} the norm read "
      f"{armB['norm'][K_INJ] / med_pre:.0f}x its settled median BEFORE optimizer.step() "
      f"was taken -- the one moment the update could still have been refused; every "
      f"loss that could reveal the DAMAGE (rather than the weird batch) existed only "
      f"afterwards, and stayed elevated for {rec_B} steps.")
print(f"AdamW honest note: the identical incident under AdamW moved the probe by only "
      f"+{worst_E - preE:.4f} nats (vs +{worst_B - pre_probe:.4f} under SGD) -- the "
      f"per-coordinate preconditioner absorbs most of a one-step spike before clipping "
      f"even enters; clipping is the guard for what it cannot absorb.")

saved plots/gradnorm_lead.png

the honest sentence: at step 150 the norm read 16x its settled median BEFORE optimizer.step() was taken -- the one moment the update could still have been refused; every loss that could reveal the DAMAGE (rather than the weird batch) existed only afterwards, and stayed elevated for 17 steps.
AdamW honest note: the identical incident under AdamW moved the probe by only +0.1459 nats (vs +0.6543 under SGD) -- the per-coordinate preconditioner absorbs most of a one-step spike before clipping even enters; clipping is the guard for what it cannot absorb.


## §7 — Task 5: my own MFU, computed honestly

Three numbers make an MFU, and each hides a decision that must be pinned:

1. **FLOPs per token.** The famous 6N is a *convention*, not a fact about the model.
   For this architecture the exact matmul count is
   `6·(12·L·d² + V·d) + 12·L·T·d` per token (linears: 2 FLOPs/MAC forward, 2× that
   backward; attention's QKᵀ and AV matmuls: the 12·L·T·d term that 6N ignores). The
   cell below scores four 6N-style conventions against the exact count — and shows the
   popular 6·N_total lands within 1% *only by accidental cancellation*: the position
   table and LayerNorms (zero-FLOP parameters) overcount by almost exactly what the
   ignored attention term undercounts, at this T. Change T and the "convention" drifts
   by tens of percent. A profiled forward pass (torch counts 2mnk per matmul) then
   cross-checks the analytic count against what actually executes.
2. **Tokens per second.** Total *processed* tokens (B·T = 1,024 per step; 1,016 of
   them scored) over total wall-clock of a post-warmup window — not a median of step
   times, which quietly deletes real recurring costs.
3. **Peak FLOP/s.** No datasheet exists for a slice of a shared container, so the
   denominator is a *measured attainable* peak: the best sustained fp32 GEMM this
   machine will do (size-swept, best-of-repeats, interleaved with the training
   measurement so drift hits both). Named honestly: **utilization vs attainable GEMM**,
   which *flatters* the number relative to the industry MFU-vs-theoretical-peak
   convention (a good GEMM reaches only ~70–90% of silicon peak) — the true MFU is
   lower than what is printed here, and the bias direction is stated wherever the
   number appears.

In [28]:
def flops_per_token_exact(d, T, L, V):
    """Matmul FLOPs per token, fwd+bwd: linears 6/MAC-col, attention 12*L*T*d."""
    linear = 12 * L * d * d + V * d          # qkv 3d^2 + proj d^2 + mlp 8d^2 -> 12Ld^2; head Vd
    return 6 * linear + 12 * L * T * d


def flops_forward_matmul(d, T, L, V):
    return 2 * (12 * L * d * d + V * d) + 4 * L * T * d


def gemm_peak_proxy(sizes, reps=3):
    """Best sustained fp32 GEMM this machine will do right now. 2n^3 FLOPs per mm."""
    best = 0.0
    for n in sizes:
        a = torch.randn(n, n); b_ = torch.randn(n, n); out = torch.empty(n, n)
        torch.mm(a, b_, out=out)                     # warm the kernel + allocation
        for _ in range(reps):
            t0 = time.perf_counter()
            torch.mm(a, b_, out=out)
            best = max(best, 2 * n ** 3 / (time.perf_counter() - t0))
    return best


def timed_training(model_d, warmup, steps, seed=SEED):
    """The plain single-batch loop (AdamW), timed with perf_counter. Returns
    (tokens_per_sec, per-step seconds, n_params)."""
    torch.manual_seed(seed)
    m = TinyLM(d=model_d, n_layer=CFG["n_layer"],
               n_head=max(2, CFG["n_head"] * model_d // CFG["d_model"]))
    opt = torch.optim.AdamW(m.parameters(), lr=CFG["lr"])
    rng = random.Random(seed + 1)
    batches = [torch.tensor([prose_crop(rng) for _ in range(CFG["B"])])
               for _ in range(warmup + steps)]
    per_step = []
    for i, batch in enumerate(batches):
        t0 = time.perf_counter()
        opt.zero_grad(set_to_none=True)
        token_weighted_loss(m(batch), batch).backward()
        opt.step()
        if i >= warmup:
            per_step.append(time.perf_counter() - t0)
    toks = CFG["B"] * CFG["T"] * len(per_step)
    return toks / sum(per_step), per_step, count_params(m)


In [29]:
d, T, L, V = CFG["d_model"], CFG["T"], CFG["n_layer"], VOCAB
EXACT = flops_per_token_exact(d, T, L, V)
n_tok_emb = V * d
n_pos_emb = CFG["max_pos"] * d
conventions = {
    "6 x N_total": 6 * N_PARAMS,
    "6 x (N - tok_emb)": 6 * (N_PARAMS - n_tok_emb),
    "6 x (N - both embeddings)": 6 * (N_PARAMS - n_tok_emb - n_pos_emb),
    "6 x N_matmul (no attention term)": 6 * (12 * L * d * d + V * d),
}
print(f"exact matmul FLOPs/token (fwd+bwd) = 6*(12Ld^2 + Vd) + 12LTd = {EXACT:,}")
conv_rows = {}
for name, val in conventions.items():
    err = (val - EXACT) / EXACT * 100
    conv_rows[name] = {"flops": val, "err_pct": err}
    print(f"  {name:34s} {val:>10,}  {err:+6.2f}% vs exact")
print("\nwhy 6*N_total nearly cancels HERE (and only here):")
zero_flop = n_pos_emb + sum(p.numel() for n_, p in model.named_parameters() if "ln" in n_)
print(f"  zero-FLOP params (pos table + LayerNorms): {zero_flop:,} "
      f"(+{6 * zero_flop / EXACT * 100:.1f}% overcount)")
print(f"  ignored attention matmuls 12LTd: {12 * L * T * d:,} "
      f"(-{12 * L * T * d / EXACT * 100:.1f}% undercount)")
tdep = {}
for T_ in (32, 128, 512):
    ex = flops_per_token_exact(d, T_, L, V)
    tdep[T_] = (6 * N_PARAMS - ex) / ex * 100
    print(f"  at T={T_:3d}: 6*N_total is {tdep[T_]:+.1f}% off exact -- the cancellation is a coincidence of T")

# cross-check: torch's own per-op FLOP counter on one forward pass
from torch.profiler import profile, ProfilerActivity
probe_batch = torch.tensor([prose_crop(random.Random(3)) for _ in range(CFG["B"])])
with profile(activities=[ProfilerActivity.CPU], with_flops=True) as prof:
    with torch.no_grad():
        model(probe_batch)
measured_fwd = sum(e.flops for e in prof.key_averages() if e.flops) / (CFG["B"] * CFG["T"])
analytic_fwd = flops_forward_matmul(d, T, L, V)
rel = abs(measured_fwd - analytic_fwd) / analytic_fwd
print(f"\nprofiler-measured forward matmul FLOPs/token: {measured_fwd:,.0f} "
      f"vs analytic {analytic_fwd:,}  (rel diff {rel:.2e})")
assert rel < 1e-3, "the analytic count must match what actually executes"
RESULTS["mfu_flops"] = {"exact_per_token": EXACT, "conventions": conv_rows,
                        "t_dependence": tdep, "zero_flop_params": zero_flop,
                        "profiler_fwd_per_token": measured_fwd,
                        "analytic_fwd_per_token": analytic_fwd}

exact matmul FLOPs/token (fwd+bwd) = 6*(12Ld^2 + Vd) + 12LTd = 2,951,424
  6 x N_total                         2,961,408   +0.34% vs exact
  6 x (N - tok_emb)                   2,762,496   -6.40% vs exact
  6 x (N - both embeddings)           2,565,888  -13.06% vs exact
  6 x N_matmul (no attention term)    2,558,208  -13.32% vs exact

why 6*N_total nearly cancels HERE (and only here):
  zero-FLOP params (pos table + LayerNorms): 34,048 (+6.9% overcount)
  ignored attention matmuls 12LTd: 393,216 (-13.3% undercount)
  at T= 32: 6*N_total is +11.5% off exact -- the cancellation is a coincidence of T
  at T=128: 6*N_total is +0.3% off exact -- the cancellation is a coincidence of T
  at T=512: 6*N_total is -28.3% off exact -- the cancellation is a coincidence of T

profiler-measured forward matmul FLOPs/token: 984,448 vs analytic 983,808  (rel diff 6.51e-04)


USDT:2026-08-29 09:33:58 2653:2653 SyncActivityProfilerHandler.cpp:52] profiler_start
USDT:2026-08-29 09:33:58 2653:2653 SyncActivityProfilerHandler.cpp:59] profiler_stop


In [30]:
# ---- the measurement: proxy / tps / proxy / instrumented run / proxy ----
proxy1 = gemm_peak_proxy(CFG["gemm_sizes"])
tps, per_step, n_par = timed_training(CFG["d_model"], CFG["mfu_warmup"], CFG["mfu_steps"])
proxy2 = gemm_peak_proxy(CFG["gemm_sizes"])
achieved = EXACT * tps
PEAK = max(proxy1, proxy2)
mfu = achieved / PEAK
mfu_6n = 6 * N_PARAMS * tps / PEAK
step_ms = sorted(per_step)[len(per_step) // 2] * 1e3
iqr = (sorted(per_step)[3 * len(per_step) // 4] - sorted(per_step)[len(per_step) // 4]) * 1e3

print(f"GEMM peak proxy (before/after): {proxy1 / 1e9:.0f} / {proxy2 / 1e9:.0f} GFLOP/s "
      f"-> denominator {PEAK / 1e9:.0f} GFLOP/s")
print(f"training: {tps:,.0f} processed tokens/s "
      f"({CFG['B'] * CFG['T']} per step; median step {step_ms:.1f} ms, IQR {iqr:.1f} ms)")
print(f"achieved: {EXACT:,} FLOPs/token x {tps:,.0f} tok/s = {achieved / 1e9:.1f} GFLOP/s")
print(f"\nMFU (vs attainable GEMM) = {mfu * 100:.1f}%   [6N convention: {mfu_6n * 100:.1f}%]")
print("vs true theoretical silicon peak (unknowable on a shared container, and above "
      "any GEMM): the honest MFU is LOWER than this number.")
assert 0 < mfu < 1 and achieved <= PEAK, "an MFU above the measured peak means the accounting is broken"

# the session's worked example, every assumption pinned
ach_ex = 6 * 9e9 * 12_000
peak_ex = 8 * 989e12          # 8 x H100 SXM, DENSE bf16 (989 TFLOP/s each)
mfu_ex = ach_ex / peak_ex
print(f"\nsession worked example: 6 x 9e9 x 12,000 = {ach_ex / 1e12:.0f} TFLOP/s over "
      f"8 x 989 TFLOP/s (H100 SXM dense bf16) = {mfu_ex * 100:.4f}% ~ 8.2%")
print(f"(quoting NVIDIA's 2:4-sparsity 1,979 number instead would read "
      f"{ach_ex / (8 * 1979e12) * 100:.1f}% -- denominator choice is half of any MFU claim)")
assert abs(mfu_ex - 0.081901) < 1e-4
RESULTS["mfu_main"] = {"proxy_before": proxy1, "proxy_after": proxy2, "peak": PEAK,
                       "tps": tps, "achieved": achieved, "mfu": mfu, "mfu_6n": mfu_6n,
                       "step_ms_median": step_ms, "step_ms_iqr": iqr,
                       "per_step_s": per_step,
                       "worked_example": {"achieved": ach_ex, "peak": peak_ex,
                                          "mfu": mfu_ex}}
HEADLINE["mfu"] = f"{mfu * 100:.1f}"
HEADLINE["tps"] = f"{tps:,.0f}"
HEADLINE["peak_gflops"] = f"{PEAK / 1e9:.0f}"

GEMM peak proxy (before/after): 471 / 471 GFLOP/s -> denominator 471 GFLOP/s
training: 39,074 processed tokens/s (1024 per step; median step 25.5 ms, IQR 1.4 ms)
achieved: 2,951,424 FLOPs/token x 39,074 tok/s = 115.3 GFLOP/s

MFU (vs attainable GEMM) = 24.5%   [6N convention: 24.6%]
vs true theoretical silicon peak (unknowable on a shared container, and above any GEMM): the honest MFU is LOWER than this number.

session worked example: 6 x 9e9 x 12,000 = 648 TFLOP/s over 8 x 989 TFLOP/s (H100 SXM dense bf16) = 8.1901% ~ 8.2%
(quoting NVIDIA's 2:4-sparsity 1,979 number instead would read 4.1% -- denominator choice is half of any MFU claim)


### 7b. Where the other ~90+% goes — measured, not folklore

The GPU explanations (kernel launches, missing fusion) mostly do not apply on CPU at
these sizes; what does is measurable with the op-level profiler: how much of the step
even *runs* FLOP-counted matmuls, and how far below the 4096² GEMM rate those small
matmuls sit. Then the width sweep makes the diagnosis concrete: same code, same T, same
B, only d grows — and utilization climbs all the way into the "healthy" band, because
big matrices amortize everything that is not a matmul. The distance to 40% is not
mystery overhead; it is **matrices that are too small**.

In [31]:
from torch.profiler import profile as _profile
seed_all()
prof_model = TinyLM()
prof_opt = torch.optim.AdamW(prof_model.parameters(), lr=CFG["lr"])
prof_batch = torch.tensor([prose_crop(random.Random(5)) for _ in range(CFG["B"])])
for _ in range(3):    # warm
    prof_opt.zero_grad(set_to_none=True)
    token_weighted_loss(prof_model(prof_batch), prof_batch).backward()
    prof_opt.step()
with _profile(activities=[ProfilerActivity.CPU]) as prof:
    for _ in range(5):
        prof_opt.zero_grad(set_to_none=True)
        token_weighted_loss(prof_model(prof_batch), prof_batch).backward()
        prof_opt.step()
MATMUL_OPS = {"aten::mm", "aten::addmm", "aten::bmm", "aten::matmul"}
rows = sorted(prof.key_averages(), key=lambda e: e.self_cpu_time_total, reverse=True)
total_us = sum(e.self_cpu_time_total for e in rows)
matmul_us = sum(e.self_cpu_time_total for e in rows if e.key in MATMUL_OPS)
print(f"{'op':34s} {'self time':>10s}  {'share':>6s}  counted in FLOPs?")
op_table = []
for e in rows[:14]:
    share = e.self_cpu_time_total / total_us * 100
    counted = "YES" if e.key in MATMUL_OPS else "no"
    op_table.append({"op": e.key, "share_pct": share, "counted": counted == "YES"})
    print(f"{e.key:34s} {e.self_cpu_time_total / 1e3:8.1f}ms  {share:5.1f}%  {counted}")
matmul_share = matmul_us / total_us
print(f"\nFLOP-counted matmul ops get {matmul_share * 100:.0f}% of the step's CPU time; "
      f"softmax, layernorm, gelu, Adam and glue get the rest --")
print(f"an upper bound of {matmul_share * 100:.0f}% x (small-matmul efficiency) "
      f"before a single FLOP is 'wasted'. Measured MFU {mfu * 100:.1f}% implies the "
      f"matmuls themselves run at ~{mfu / matmul_share * 100:.0f}% of the 4096^2 GEMM rate.")
RESULTS["mfu_profile"] = {"matmul_time_share": matmul_share, "ops": op_table}
HEADLINE["matmul_share"] = f"{matmul_share * 100:.0f}"

USDT:2026-08-29 09:34:03 2653:2653 SyncActivityProfilerHandler.cpp:52] profiler_start
USDT:2026-08-29 09:34:03 2653:2653 SyncActivityProfilerHandler.cpp:59] profiler_stop


op                                  self time   share  counted in FLOPs?
aten::mm                               49.9ms   38.4%  YES
aten::bmm                               8.3ms    6.4%  YES
aten::copy_                             6.6ms    5.1%  no
Optimizer.step#AdamW.step               5.6ms    4.3%  no
aten::gelu_backward                     5.1ms    3.9%  no
aten::masked_fill_                      4.2ms    3.2%  no
aten::native_layer_norm_backward        4.0ms    3.1%  no
aten::div                               3.8ms    3.0%  no
aten::mul_                              3.2ms    2.4%  no
aten::sqrt                              2.5ms    1.9%  no
aten::gelu                              2.2ms    1.7%  no
aten::_softmax                          2.0ms    1.6%  no
aten::lerp_                             2.0ms    1.5%  no
aten::add_                              1.9ms    1.5%  no

FLOP-counted matmul ops get 45% of the step's CPU time; softmax, layernorm, gelu, Adam and glue get the rest --


In [32]:
sweep = []
for d_ in CFG["widths"]:
    tps_, steps_, n_ = timed_training(d_, max(2, CFG["mfu_warmup"] // 2),
                                      max(8, CFG["mfu_steps"] // 2))
    ex_ = flops_per_token_exact(d_, T, L, V)
    sweep.append({"d": d_, "params": n_, "flops_per_token": ex_,
                  "ms_per_step": sum(steps_) / len(steps_) * 1e3,
                  "tps": tps_, "mfu": ex_ * tps_ / PEAK})
proxy3 = gemm_peak_proxy(CFG["gemm_sizes"])
print(f"{'d':>4s} {'params':>10s} {'FLOPs/tok':>11s} {'ms/step':>8s} {'tok/s':>8s} {'MFU':>6s}")
for z in sweep:
    print(f"{z['d']:4d} {z['params']:10,d} {z['flops_per_token']:11,d} "
          f"{z['ms_per_step']:8.1f} {z['tps']:8,.0f} {z['mfu'] * 100:5.1f}%")
print(f"(GEMM proxy re-measured after the sweep: {proxy3 / 1e9:.0f} GFLOP/s)")
strict_assert(sweep[-1]["mfu"] > sweep[0]["mfu"], "utilization must rise with width")
RESULTS["mfu_sweep"] = {"rows": sweep, "proxy_after_sweep": proxy3}
HEADLINE["mfu_wide"] = f"{sweep[-1]['mfu'] * 100:.1f}"
HEADLINE["mfu_narrow"] = f"{sweep[0]['mfu'] * 100:.1f}"

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot([z["d"] for z in sweep], [z["mfu"] * 100 for z in sweep], "o-")
axes[0].axhspan(35, 50, color="tab:green", alpha=0.12, label="'healthy' 35-50% band")
axes[0].set_xlabel("model width d"); axes[0].set_ylabel("utilization vs attainable GEMM (%)")
axes[0].set_title("same loop, wider matrices"); axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)
axes[1].bar(["matmul ops\n(FLOP-counted)", "everything else\n(softmax, LN, gelu,\nAdam, glue)"],
            [matmul_share * 100, (1 - matmul_share) * 100],
            color=["tab:blue", "tab:orange"])
axes[1].set_ylabel("share of step CPU time (%)")
axes[1].set_title(f"where the d={CFG['d_model']} step actually goes")
fig.tight_layout(); fig.savefig(ART / "plots" / "mfu.png", dpi=120)
plt.close(fig); print("saved plots/mfu.png")

   d     params   FLOPs/tok  ms/step    tok/s    MFU
  64    148,480     885,888     16.8   60,826  11.4%
 128    493,568   2,951,424     26.2   39,109  24.5%
 256  1,773,568  10,621,440     71.0   14,417  32.5%
 512  6,692,864  40,117,248    173.9    5,889  50.2%
(GEMM proxy re-measured after the sweep: 467 GFLOP/s)


saved plots/mfu.png


### 7c. The four traces, on one screen

The dashboard the session prescribes — loss, grad norm, tokens/sec, MFU — logged per
step on one clean instrumented run. The instrumentation itself (norm + timing + the
extra bookkeeping) costs throughput, which is exactly why the headline tokens/sec above
came from an *uninstrumented* loop; the gap between the two is printed rather than
hidden.

In [33]:
seed_all()
tr_model = TinyLM()
tr_opt = torch.optim.AdamW(tr_model.parameters(), lr=CFG["lr"])
rng = random.Random(11)
n_tr = CFG["steps_norm"]
four = {"loss": [], "norm": [], "tps": [], "mfu": []}
for s in range(n_tr):
    batch = torch.tensor([prose_crop(rng) for _ in range(CFG["B"])])
    t0 = time.perf_counter()
    tr_opt.zero_grad(set_to_none=True)
    loss = token_weighted_loss(tr_model(batch), batch)
    loss.backward()
    norm = float(torch.nn.utils.clip_grad_norm_(tr_model.parameters(), float("inf")))
    tr_opt.step()
    dt = time.perf_counter() - t0
    four["loss"].append(float(loss.detach())); four["norm"].append(norm)
    four["tps"].append(CFG["B"] * CFG["T"] / dt)
    four["mfu"].append(EXACT * CFG["B"] * CFG["T"] / dt / PEAK)
inst_tps = sum(four["tps"][5:]) / len(four["tps"][5:])
print(f"instrumented loop: {inst_tps:,.0f} tok/s vs plain loop {tps:,.0f} tok/s "
      f"({(1 - inst_tps / tps) * 100:+.0f}% instrumentation cost)")
CURVES["four_traces"] = four
RESULTS["mfu_main"]["instrumented_tps"] = inst_tps

fig, axes = plt.subplots(4, 1, figsize=(9, 8), sharex=True)
for ax_, key, color, label in [
        (axes[0], "loss", "tab:blue", "train loss (nats)"),
        (axes[1], "norm", "tab:red", "grad norm (pre-clip)"),
        (axes[2], "tps", "tab:green", "tokens / second"),
        (axes[3], "mfu", "tab:purple", "MFU vs attainable GEMM")]:
    ax_.plot(range(n_tr), four[key], color=color, lw=0.9)
    ax_.set_ylabel(label, fontsize=8); ax_.grid(alpha=0.3)
axes[3].set_xlabel("step")
axes[0].set_title("the four traces: is it learning / is it in trouble / did it get "
                  "slower / is the machine wasted")
fig.tight_layout(); fig.savefig(ART / "plots" / "four_traces.png", dpi=120)
plt.close(fig); print("saved plots/four_traces.png")

instrumented loop: 37,216 tok/s vs plain loop 39,074 tok/s (+5% instrumentation cost)


saved plots/four_traces.png


## §8 — Task 6: the number 0.1, written out by hand

A float is three fields — sign, exponent (how big), mantissa (which number) — and
0.1 is the perfect stress test because in binary it is **infinite**:
0.000110011001100…₂, the pattern `1100` forever. Every format must cut it somewhere,
and where it cuts is the whole story.

The derivation, once, by hand (the code below repeats it exactly, in exact rational
arithmetic, for every format — and is cross-checked bit-for-bit against torch):

1. expand: 0.1 = 0.0001100110011…₂;
2. normalize: 0.1 = 1.100110011…₂ × 2⁻⁴ (slide the point 4 places; the leading 1 is
   *implicit* — stored for free);
3. exponent field = −4 + bias (127 for fp32/bf16 → `01111011`; 15 for fp16 → `01011`;
   7 for fp8 E4M3 → `0011`);
4. mantissa = the first M bits of `10011001100…`, rounded to nearest, **ties to
   even** — the dropped tail here is never exactly half a ULP (the expansion repeats
   forever), so 0.1 exercises the "nearest" but never the "even" (dedicated tie
   probes below exercise that);
5. reassemble and read back what got stored.

One pitfall the code refuses to step on: `Fraction(0.1)` is **not** 1/10 — it is the
float64 number nearest 1/10 (a 55-bit-denominator rational). Every error below is
measured against `Fraction(1, 10)`, the exact value. (For 0.1 the two references agree
to well past every format's precision, and torch's decimal→fp64→fp32→target path is
asserted to coincide with rounding the exact 1/10 — but only because it is checked.)

In [34]:
class FloatSpec:
    """sign + exp_bits + mant_bits, IEEE-754-style; e4m3fn = the OCP/NVIDIA variant
    (bias 7, NO infinities, exponent 1111 carries finite values except mantissa 111
    which is NaN; torch saturates on overflow)."""

    def __init__(self, name, exp_bits, mant_bits, e4m3fn=False):
        self.name, self.E, self.M, self.e4m3fn = name, exp_bits, mant_bits, e4m3fn
        self.bias = (1 << (exp_bits - 1)) - 1

    def max_finite_exp_field(self):
        return ((1 << self.E) - 1) if self.e4m3fn else ((1 << self.E) - 2)


HALF = Fraction(1, 2)


def rne(scaled):
    """Round an exact non-negative Fraction to the nearest integer, ties to even."""
    lo = scaled.numerator // scaled.denominator
    rem = scaled - lo
    if rem > HALF or (rem == HALF and lo % 2 == 1):
        lo += 1
    return lo


def encode_float(x, spec):
    """Exact Fraction -> (bits, stored Fraction, info). From first principles:
    exact exponent search, RNE via exact remainder, mantissa carry, subnormals."""
    assert isinstance(x, Fraction)
    sign = 0 if x >= 0 else 1
    a = abs(x)
    E, M, bias = spec.E, spec.M, spec.bias
    info = {"subnormal": False, "underflow": False, "overflow": False}
    if a == 0:
        return f"0|{'0' * E}|{'0' * M}", Fraction(0), info
    e, t = 0, Fraction(1)
    if a >= 1:
        while t * 2 <= a:
            t *= 2; e += 1
    else:
        while a < t:
            t /= 2; e -= 1
    exp_field = e + bias
    if exp_field <= 0:                                  # subnormal territory
        info["subnormal"] = True
        mant = rne(a / Fraction(2) ** (1 - bias) * (1 << M))
        if mant == 0:
            info["underflow"] = True
            exp_field, mant, stored = 0, 0, Fraction(0)
        elif mant >= (1 << M):                          # rounded up into min normal
            info["subnormal"] = False
            exp_field, mant = 1, 0
            stored = Fraction(2) ** (1 - bias)
        else:
            exp_field = 0
            stored = Fraction(mant, 1 << M) * Fraction(2) ** (1 - bias)
    else:
        mant = rne((a / t - 1) * (1 << M))
        if mant == (1 << M):                            # carry: 1.111.. rounds to 10.0
            mant, exp_field = 0, exp_field + 1
        if exp_field > spec.max_finite_exp_field() or (
                spec.e4m3fn and exp_field == spec.max_finite_exp_field()
                and mant == (1 << M) - 1):              # would collide with e4m3fn NaN
            info["overflow"] = True
            stored = None
        else:
            stored = (1 + Fraction(mant, 1 << M)) * Fraction(2) ** (exp_field - bias)
    if sign:
        stored = -stored if stored is not None else None
    bits = f"{sign}|{exp_field:0{E}b}|{mant:0{M}b}"
    return bits, stored, info


FP32 = FloatSpec("fp32", 8, 23)
BF16 = FloatSpec("bf16", 8, 7)
FP16 = FloatSpec("fp16", 5, 10)
FP8E4M3 = FloatSpec("fp8 E4M3", 4, 3, e4m3fn=True)


In [35]:
TENTH = Fraction(1, 10)
assert Fraction(1, 10) != Fraction(0.1), "Fraction(0.1) is the float64 impostor"
print(f"pitfall, shown: Fraction(0.1) = {Fraction(0.1).numerator}/{Fraction(0.1).denominator}")
print(f"                Fraction(1,10) = 1/10 -- all errors below measure against THIS\n")

# the binary expansion, by repeated doubling (the by-hand algorithm)
bits_, xx = [], TENTH
for _ in range(20):
    xx *= 2
    bits_.append(1 if xx >= 1 else 0)
    if xx >= 1:
        xx -= 1
print(f"0.1 in binary: 0.{''.join(map(str, bits_))}... (the pattern 1100 repeats forever)")
print(f"normalize    : 1.{''.join(map(str, bits_[3:]))}... x 2^-4\n")

def torch_bits(val, dtype, nbits):
    t_ = torch.tensor(val, dtype=torch.float32).to(dtype)
    iv = t_.view({16: torch.int16, 8: torch.uint8}[nbits]).item() & ((1 << nbits) - 1)
    return f"{iv:0{nbits}b}", float(t_)

fmt_rows = []
for spec, tdtype, nbits in [(FP32, torch.float32, None), (BF16, torch.bfloat16, 16),
                            (FP16, torch.float16, 16), (FP8E4M3, torch.float8_e4m3fn, 8)]:
    bits, stored, info = encode_float(TENTH, spec)
    abs_err = abs(stored - TENTH)
    rel_err = abs_err / TENTH
    # bit-for-bit against torch (fp32 via struct; the rest via bit reinterpretation)
    if tdtype is torch.float32:
        tb = format(struct.unpack(">I", struct.pack(">f", 0.1))[0], "032b")
        tv = struct.unpack(">f", struct.pack(">f", 0.1))[0]
    else:
        tb, tv = torch_bits(0.1, tdtype, nbits)
    assert bits.replace("|", "") == tb, f"{spec.name}: hand bits != torch bits"
    assert float(stored) == tv, f"{spec.name}: stored value != torch value"
    fmt_rows.append({"format": spec.name, "bits": bits, "stored": float(stored),
                     "stored_exact": f"{stored.numerator}/{stored.denominator}",
                     "abs_err": float(abs_err), "rel_err": float(rel_err),
                     "exp_bits": spec.E, "mant_bits": spec.M, "bias": spec.bias,
                     "rounded": "up" if stored > TENTH else "down"})
    print(f"{spec.name:9s} {bits:36s} sign|exp(-4+{spec.bias}={spec.bias - 4})|mantissa")
    print(f"{'':9s} stores {float(stored):.17g}  (rounded {fmt_rows[-1]['rounded']}; "
          f"abs err {float(abs_err):.3e}, rel err {float(rel_err):.3e})")
    print(f"{'':9s} torch agrees bit-for-bit: {tb}\n")

print("only fp16 rounds DOWN (its guard bit falls on a 0 of the 1100 pattern) -- "
      "the direction of the cut depends on where the format's edge lands, not on luck.")
RESULTS["float_bits"] = {"formats": fmt_rows}
HEADLINE["fp32_rel"] = f"{fmt_rows[0]['rel_err']:.1e}"
HEADLINE["bf16_rel"] = f"{fmt_rows[1]['rel_err']:.1e}"
HEADLINE["fp8_rel"] = f"{fmt_rows[3]['rel_err']:.1e}"

pitfall, shown: Fraction(0.1) = 3602879701896397/36028797018963968
                Fraction(1,10) = 1/10 -- all errors below measure against THIS

0.1 in binary: 0.00011001100110011001... (the pattern 1100 repeats forever)
normalize    : 1.11001100110011001... x 2^-4

fp32      0|01111011|10011001100110011001101   sign|exp(-4+127=123)|mantissa
          stores 0.10000000149011612  (rounded up; abs err 1.490e-09, rel err 1.490e-08)
          torch agrees bit-for-bit: 00111101110011001100110011001101

bf16      0|01111011|1001101                   sign|exp(-4+127=123)|mantissa
          stores 0.10009765625  (rounded up; abs err 9.766e-05, rel err 9.766e-04)
          torch agrees bit-for-bit: 0011110111001101

fp16      0|01011|1001100110                   sign|exp(-4+15=11)|mantissa
          stores 0.0999755859375  (rounded down; abs err 2.441e-05, rel err 2.441e-04)
          torch agrees bit-for-bit: 0010111001100110

fp8 E4M3  0|0011|101                           sign|exp(-4+7=3)|m

In [36]:
# ties-to-even, exercised on purpose (0.1 never ties; these do, in both directions)
tie_probes = [
    (Fraction(17, 256), FP8E4M3, torch.float8_e4m3fn, 8, 0.0625,   "down to even mantissa 000"),
    (Fraction(19, 256), FP8E4M3, torch.float8_e4m3fn, 8, 0.078125, "up to even mantissa 010"),
    (Fraction(257, 256), BF16, torch.bfloat16, 16, 1.0,            "down to even (1.0)"),
    (Fraction(259, 256), BF16, torch.bfloat16, 16, 1.015625,       "up to even"),
]
tie_rows = []
for frac, spec, tdtype, nbits, expected, note in tie_probes:
    bits, stored, _ = encode_float(frac, spec)
    tb, tv = torch_bits(float(frac), tdtype, nbits)
    assert float(stored) == expected == tv and bits.replace("|", "") == tb
    tie_rows.append({"value": f"{frac.numerator}/{frac.denominator}",
                     "format": spec.name, "stored": float(stored), "note": note})
    print(f"{spec.name:9s} {str(frac):8s} -> {float(stored):<10.6g} ({note}; torch agrees)")
print("=> the encoder and torch both round to nearest, TIES TO EVEN, verified in both directions.")
RESULTS["float_bits"]["tie_probes"] = tie_rows
print(f"\nE4M3FN is the OCP variant, not IEEE: no infinities, NaN = S.1111.111, and torch "
      f"SATURATES on overflow: fp8(500.0) = "
      f"{float(torch.tensor(500.0).to(torch.float8_e4m3fn))}")
assert float(torch.tensor(500.0).to(torch.float8_e4m3fn)) == 448.0

fp8 E4M3  17/256   -> 0.0625     (down to even mantissa 000; torch agrees)
fp8 E4M3  19/256   -> 0.078125   (up to even mantissa 010; torch agrees)
bf16      257/256  -> 1          (down to even (1.0); torch agrees)
bf16      259/256  -> 1.01562    (up to even; torch agrees)
=> the encoder and torch both round to nearest, TIES TO EVEN, verified in both directions.

E4M3FN is the OCP variant, not IEEE: no infinities, NaN = S.1111.111, and torch SATURATES on overflow: fp8(500.0) = 448.0


### 8b. Which one to train in — the demos that decide it

The session's table said fp16 loses gradients at 10⁻⁸ while bf16 keeps them. The exact
boundary matters and is measured here: fp16's smallest *subnormal* is 2⁻²⁴ ≈ 5.96×10⁻⁸,
and under round-to-nearest anything below **half** of it (2⁻²⁵ ≈ 2.98×10⁻⁸) becomes
exactly zero — a weight whose gradient sits there simply stops learning, silently.
Loss scaling (×1024 before backward, ÷1024 after) rescues *representability*, not
precision: the rescued value lands in fp16's subnormal range carrying ~7 effective
mantissa bits, and anything below ~3×10⁻¹¹ still dies. bf16 needs none of that
machinery because it keeps fp32's full exponent — at the price this section already
measured: 9.8×10⁻⁴ relative error on 0.1, four thousand times coarser than fp32.
(Subnormal behavior is CPU-measured here; some accelerators flush subnormals to zero,
which makes fp16's cliff *worse*, not better.)

In [37]:
cliff = []
for gval, note in [(1e-4, "well above the cliff"), (1e-6, "losing digits"),
                   (4e-8, "rounds UP to the smallest subnormal"),
                   (3e-8, "still rounds up (just above half the subnormal)"),
                   (2.9e-8, "below half the smallest subnormal -> ZERO"),
                   (1e-8, "the session's row -> ZERO")]:
    f16 = float(torch.tensor(gval, dtype=torch.float32).to(torch.float16))
    b16 = float(torch.tensor(gval, dtype=torch.float32).to(torch.bfloat16))
    rel16 = abs(f16 - gval) / gval
    relb = abs(b16 - gval) / gval
    cliff.append({"g": gval, "fp16": f16, "bf16": b16,
                  "fp16_rel": rel16, "bf16_rel": relb})
    print(f"gradient {gval:8.1e}: fp16 -> {f16:12.6e} (rel err {rel16:7.1e})   "
          f"bf16 -> {b16:12.6e} (rel err {relb:.1e})   {note}")
assert cliff[-1]["fp16"] == 0.0 and cliff[-2]["fp16"] == 0.0
assert cliff[2]["fp16"] != 0.0 and cliff[-1]["bf16"] != 0.0
print(f"\nfp16 boundary: min subnormal 2^-24 = {2 ** -24:.3e}; "
      f"round-to-zero cutoff 2^-25 = {2 ** -25:.3e}")

scaled = float(torch.tensor(1e-8 * 1024, dtype=torch.float32).to(torch.float16))
rel_scaled = abs(scaled / 1024 - 1e-8) / 1e-8
dead_even_scaled = float(torch.tensor(1e-11 * 1024, dtype=torch.float32).to(torch.float16))
print(f"loss scaling x1024: 1e-8 -> 1.024e-5 -> fp16 {scaled:.6e} -> unscale "
      f"{scaled / 1024:.6e} (rel err {rel_scaled:.1e}: representable, in the "
      f"SUBNORMAL range, ~7 effective bits)")
print(f"but 1e-11 x 1024 = {dead_even_scaled} in fp16 -- the cliff moved, it did not go away")
assert scaled != 0.0 and dead_even_scaled == 0.0
RESULTS["float_bits"]["cliff"] = cliff
RESULTS["float_bits"]["loss_scaling"] = {"scaled_1e8": scaled, "rel_err": rel_scaled,
                                         "dead_1e11": dead_even_scaled}

# the encoder walks the same cliff from first principles (subnormal path exercised)
bits_u, stored_u, info_u = encode_float(Fraction(1, 10 ** 8), FP16)
bits_s, stored_s, info_s = encode_float(Fraction(3, 10 ** 8), FP16)
print(f"\nencoder, from first principles: fp16(1e-8) = {bits_u} "
      f"(underflow={info_u['underflow']}), fp16(3e-8) = {bits_s} -> {float(stored_s):.3e} "
      f"(subnormal={info_s['subnormal']})")
assert info_u["underflow"] and float(stored_u) == 0.0
assert info_s["subnormal"] and float(stored_s) == 2 ** -24

print("""
THE CHOICE -- a recommendation these demos support, not something this fp32 CPU
notebook trained end-to-end: **bf16**, because
  1. range beats digits for gradients: fp16's zero-cliff at 3e-8 silences exactly the
     weights whose learning signal is faintest (measured above); bf16 shares fp32's
     exponent and never needs the loss-scaling machinery or its residual cliff;
  2. the digits bf16 gives up (rel err 9.8e-4 on 0.1, vs fp16's 2.4e-4) are digits the
     TRAINING STATE does not rely on: the fp32 master copy in SS9's 16 bytes/weight
     holds the precision; bf16 only has to carry activations and gradients;
  3. fp8 E4M3 (rel err 1.6e-2 on 0.1) is the 2026 production recipe WITH per-block
     scaling and higher-precision attention -- the right next step, not the safe default.
This is Session 10's own conclusion, re-derived from the bits: bf16 won because range
mattered more than digits.""")

gradient  1.0e-04: fp16 -> 1.000166e-04 (rel err 1.7e-04)   bf16 -> 1.001358e-04 (rel err 1.4e-03)   well above the cliff
gradient  1.0e-06: fp16 -> 1.013279e-06 (rel err 1.3e-02)   bf16 -> 9.983778e-07 (rel err 1.6e-03)   losing digits
gradient  4.0e-08: fp16 -> 5.960464e-08 (rel err 4.9e-01)   bf16 -> 4.004687e-08 (rel err 1.2e-03)   rounds UP to the smallest subnormal
gradient  3.0e-08: fp16 -> 5.960464e-08 (rel err 9.9e-01)   bf16 -> 3.003515e-08 (rel err 1.2e-03)   still rounds up (just above half the subnormal)
gradient  2.9e-08: fp16 -> 0.000000e+00 (rel err 1.0e+00)   bf16 -> 2.898742e-08 (rel err 4.3e-04)   below half the smallest subnormal -> ZERO
gradient  1.0e-08: fp16 -> 0.000000e+00 (rel err 1.0e+00)   bf16 -> 1.001172e-08 (rel err 1.2e-03)   the session's row -> ZERO

fp16 boundary: min subnormal 2^-24 = 5.960e-08; round-to-zero cutoff 2^-25 = 2.980e-08
loss scaling x1024: 1e-8 -> 1.024e-5 -> fp16 1.025200e-05 -> unscale 1.001172e-08 (rel err 1.2e-03: representable, in t

## §9 — What one step of training holds: 16 bytes per weight, measured

Running a finished model needs its weights. *Training* it, the way real mixed-precision
runs do, holds five tensors per weight — built here for real, byte-counted exactly, and
then extrapolated to the sizes the session quotes. This is the baseline every memory
decision in Sessions 11–13 starts from.

In [38]:
seed_all()
mem_model = TinyLM()
# the five residents of a mixed-precision training step, materialized:
w_bf16 = [p.detach().to(torch.bfloat16) for p in mem_model.parameters()]      # 2 B
g_bf16 = [torch.zeros_like(w) for w in w_bf16]                                # 2 B
master = [p.detach().float().clone().requires_grad_(True)
          for p in mem_model.parameters()]                                    # 4 B
mopt = torch.optim.AdamW(master, lr=CFG["lr"])
for m_ in master:
    m_.grad = torch.randn_like(m_) * 1e-3
mopt.step()                                                                    # moments now exist
moments = [mopt.state[m_][k] for m_ in master for k in ("exp_avg", "exp_avg_sq")]  # 8 B

rows = [("bf16 weight", w_bf16), ("bf16 gradient", g_bf16),
        ("fp32 master copy", master), ("Adam exp_avg + exp_avg_sq", moments)]
total = 0
for label, ts in rows:
    nbytes = sum(t.numel() * t.element_size() for t in ts)
    total += nbytes
    print(f"{label:28s} {nbytes:12,d} bytes  ({nbytes / N_PARAMS:.0f} B/weight)")
per_weight = total / N_PARAMS
print(f"{'TOTAL':28s} {total:12,d} bytes  = {per_weight:.0f} bytes per weight, exactly")
assert total == 16 * N_PARAMS

print(f"\nthe same 16 bytes, at real sizes (before a single activation is stored):")
scale_rows = []
for n_ in (2e9, 9e9, 20e9, 120e9):
    gib = 16 * n_ / 2 ** 30
    scale_rows.append({"params": n_, "gib": gib})
    print(f"  {n_ / 1e9:5.0f}B model -> {gib:8.1f} GiB of training state")
ceiling = 80 * 2 ** 30 / 16
print(f"  an 80 GB accelerator holds the training state of a "
      f"~{ceiling / 1e9:.1f}B model, with nothing left for activations")
print("  (which is why activation checkpointing exists: ~30% more compute for the memory back)")
RESULTS["memory"] = {"total_bytes": total, "per_weight": per_weight,
                     "scale_rows": scale_rows, "ceiling_80gb": ceiling}
HEADLINE["bytes_per_weight"] = f"{per_weight:.0f}"

bf16 weight                       987,136 bytes  (2 B/weight)
bf16 gradient                     987,136 bytes  (2 B/weight)
fp32 master copy                1,974,272 bytes  (4 B/weight)
Adam exp_avg + exp_avg_sq       3,948,544 bytes  (8 B/weight)
TOTAL                           7,897,088 bytes  = 16 bytes per weight, exactly

the same 16 bytes, at real sizes (before a single activation is stored):
      2B model ->     29.8 GiB of training state
      9B model ->    134.1 GiB of training state
     20B model ->    298.0 GiB of training state
    120B model ->   1788.1 GiB of training state
  an 80 GB accelerator holds the training state of a ~5.4B model, with nothing left for activations
  (which is why activation checkpointing exists: ~30% more compute for the memory back)


## §10 — Artifacts, and what this settles

Three of the session's V5 decisions are no longer opinions in this repo — they are
measurements: **loss is normalized by token, never by micro-batch** (§5's gap is what
the alternative costs); **the grad norm is logged from step one and the clip cap comes
from its observed distribution** (§6's cap was 2× the settled median, not a habit);
**every memory plan starts from 16 bytes per weight** (§9, byte-exact). The open
questions (training precision beyond bf16-vs-fp8, the exact cap value at scale, an
acceptable MFU, checkpointing granularity) stay open — this notebook only sharpens the
instruments that will settle them.

In [39]:
RESULTS["headline"] = HEADLINE
RESULTS["wall_seconds"] = time.time() - T0
with open(ART / "results.json", "w") as fh:
    json.dump(RESULTS, fh, indent=1, default=float)
with open(ART / "curves.json", "w") as fh:
    json.dump(CURVES, fh, default=float)
with open(ART / "run_config.json", "w") as fh:
    json.dump(RESULTS["config"], fh, indent=1, default=float)
print(f"artifacts written to {ART}/ in {RESULTS['wall_seconds']:.0f}s:")
for p_ in sorted(ART.rglob("*")):
    if p_.is_file():
        print(f"  {p_.relative_to(ART)}  ({p_.stat().st_size / 1024:.0f} KiB)")
print(f"\n{len(HEADLINE)} headline numbers; audit.py re-derives every one from disk.")
for k_, v_ in HEADLINE.items():
    print(f"  {k_:18s} {v_}")

artifacts written to submission_artifacts/ in 210s:
  curves.json  (168 KiB)
  plots/accum_gap.png  (108 KiB)
  plots/accum_negative.png  (47 KiB)
  plots/fd_ucurve.png  (57 KiB)
  plots/four_traces.png  (153 KiB)
  plots/gradnorm_lead.png  (194 KiB)
  plots/mfu.png  (64 KiB)
  results.json  (107 KiB)
  run.log  (0 KiB)
  run_config.json  (1 KiB)

29 headline numbers; audit.py re-derives every one from disk.
  n_params           493,568
  toy_grad           64.0000
  fd_rel_err         4.2e-10
  fd_decimals        10.6
  mirror_correct     2.6000
  mirror_buggy       3.0000
  mirror_err         15.4
  acc_id_f64         1.6e-16
  acc_id_bug         5.7e-01
  hz_correct         0.0424
  hz_buggy           0.0714
  hz_gap             0.0290
  hz_probe_c         0.499
  hz_probe_b         0.916
  ratio_p_star       0.6681
  norm_ratio         16.3
  dmg_B              0.6543
  dmg_C              0.1162
  scale_at_k         0.119
  mfu                24.5
  tps                39,074
  peak